In [ ]:
import json
from pathlib import Path

notebook_path = Path(
    r"C:\Users\Hp\Documents\pulseiq\PQ-02_Star_Schema_Warehouse.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("Saved PQ-02 cells:", len(nb["cells"]))
print("Saved outputs:", sum(
    len(c.get("outputs", []))
    for c in nb["cells"] if c["cell_type"] == "code"
))

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

# Project paths
PROJECT_ROOT = Path(r"C:\Users\Hp\Documents\pulseiq")
SOURCE_DB = PROJECT_ROOT / "05_Starter" / "pulseiq.duckdb"
WAREHOUSE_DB = PROJECT_ROOT / "05_Starter" / "pulseiq_warehouse.duckdb"

# Check that both databases exist
print("Project folder exists:", PROJECT_ROOT.exists())
print("Source database exists:", SOURCE_DB.exists())
print("Warehouse database exists:", WAREHOUSE_DB.exists())

assert SOURCE_DB.exists(), "Source database not found."
assert WAREHOUSE_DB.exists(), "Warehouse database not found."

# Connect to the separate warehouse database
con_dq = duckdb.connect(str(WAREHOUSE_DB))

# Attach the source database as read-only
attached_dbs = con_dq.execute("PRAGMA database_list").fetchdf()

if "source_db" not in attached_dbs["name"].astype(str).tolist():
    con_dq.execute(
        f"ATTACH '{SOURCE_DB.as_posix()}' AS source_db (READ_ONLY)"
    )

print("\nDatabase connections ready.")
display(con_dq.execute("PRAGMA database_list").fetchdf())

In [ ]:
for table_name in ["dq_checks_catalog", "dq_known_issues"]:
    print(f"\n{'=' * 60}")
    print(f"TABLE: {table_name}")
    print("=" * 60)

    df = con_dq.execute(
        f"SELECT * FROM source_db.raw.{table_name}"
    ).fetchdf()

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    display(df)

In [ ]:

# List all tables visible in the connected DuckDB databases
source_tables = con_dq.execute("SHOW ALL TABLES").fetchdf()

print("Total tables found:", len(source_tables))
display(source_tables)

# Filter for data-quality and unlock-related tables
table_name_col = next(
    (col for col in source_tables.columns if col.lower() == "table_name"),
    None
)

if table_name_col:
    possible_tables = source_tables[
        source_tables[table_name_col].astype(str).str.contains(
            "dq|quality|issue|unlock|check",
            case=False,
            na=False
        )
    ]
    print("\nPossible data-quality and unlock tables:")
    display(possible_tables)


In [ ]:

# Find PQ-03 unlock files and known-issues files
search_terms = ("unlock", "pq-03", "known_issues", "known issues")

matches = [
    path for path in PROJECT_ROOT.rglob("*")
    if path.is_file()
    and any(term in path.name.lower() for term in search_terms)
]

print("Matching files found:", len(matches))

for path in matches:
    print(path)


In [ ]:

# Read the complete data-quality check catalog
dq_catalog = con_dq.execute("""
    SELECT *
    FROM source_db.raw.dq_checks_catalog
    ORDER BY check_id
""").fetchdf()

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 50)

print("Total checks:", len(dq_catalog))
display(dq_catalog)

# Confirm the planted-issues file exists without opening its contents
issues_path = (
    PROJECT_ROOT
    / "04_Validation_Vault"
    / "unlock_PQ-03_dq_known_issues.csv"
)

print("\nPlanted-issues file exists:", issues_path.exists())


In [ ]:

# Inspect the warehouse tables used by the 14 checks
tables_to_inspect = [
    "fact_payroll",
    "fact_ledger",
    "dim_employee",
    "fact_attendance_monthly",
    "fact_leave",
    "fact_invoice",
    "fact_expense",
    "fact_task",
    "fact_attrition_risk",
]

for table_name in tables_to_inspect:
    print(f"\n{'=' * 55}")
    print(f"TABLE: warehouse.{table_name}")
    print("=" * 55)

    columns = con_dq.execute(
        f"DESCRIBE warehouse.{table_name}"
    ).fetchdf()

    display(columns[["column_name", "column_type"]])

# Check whether supporting tables exist in either database
all_tables = con_dq.execute("SHOW ALL TABLES").fetchdf()

for name in ["etl_run_log", "metric_dictionary", "dq_results"]:
    found = all_tables[all_tables["name"].str.lower() == name.lower()]
    print(f"\n{name}:")
    display(found[["database", "schema", "name"]] if not found.empty else found)


In [ ]:

# Inspect metric dictionary fields relevant to minimum group size
metric_dictionary = con_dq.execute("""
    SELECT *
    FROM source_db.raw.metric_dictionary
""").fetchdf()

print("Metric dictionary columns:", metric_dictionary.columns.tolist())
display(metric_dictionary)

# Inspect the source catalog's supporting tables
print("\nAvailable source tables:")
display(
    con_dq.execute("SHOW ALL TABLES").fetchdf()[
        ["database", "schema", "name"]
    ].query("database == 'source_db'")
)

# Preview key payroll and ledger fields
print("\nPayroll summary:")
display(con_dq.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT run_id) AS distinct_runs,
        MIN(period_end_date) AS first_period,
        MAX(period_end_date) AS last_period,
        ROUND(SUM(employer_cost), 2) AS total_employer_cost
    FROM warehouse.fact_payroll
""").fetchdf())

print("\nLedger summary:")
display(con_dq.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT entry_id) AS distinct_entries,
        MIN(entry_date) AS first_date,
        MAX(entry_date) AS last_date,
        COUNT(*) FILTER (
            WHERE account_code BETWEEN 5001 AND 5005
        ) AS salary_account_lines
    FROM warehouse.fact_ledger
""").fetchdf())


In [ ]:

# PQ-03: Initial data-quality checks (DQ-C01 to DQ-C04)

dq01 = con_dq.execute("""
    WITH payroll_monthly AS (
        SELECT
            DATE_TRUNC('month', period_end_date)::DATE AS month,
            SUM(employer_cost) AS payroll_cost
        FROM warehouse.fact_payroll
        GROUP BY 1
    ),
    ledger_monthly AS (
        SELECT
            DATE_TRUNC('month', entry_date)::DATE AS month,
            SUM(debit - credit) AS ledger_cost
        FROM warehouse.fact_ledger
        WHERE account_code BETWEEN 5001 AND 5005
        GROUP BY 1
    )
    SELECT
        COALESCE(p.month, l.month) AS month,
        p.payroll_cost,
        l.ledger_cost,
        p.payroll_cost - l.ledger_cost AS difference
    FROM payroll_monthly p
    FULL OUTER JOIN ledger_monthly l USING (month)
    WHERE p.month IS NULL
       OR l.month IS NULL
       OR ABS(p.payroll_cost - l.ledger_cost) > 0.01
    ORDER BY month
""").fetchdf()

dq02 = con_dq.execute("""
    SELECT
        entry_id,
        SUM(debit) AS total_debit,
        SUM(credit) AS total_credit,
        ABS(SUM(debit) - SUM(credit)) AS difference
    FROM warehouse.fact_ledger
    GROUP BY entry_id
    HAVING ABS(SUM(debit) - SUM(credit)) > 0.005
""").fetchdf()

dq03 = con_dq.execute("""
    SELECT
        employee_id,
        run_id,
        COUNT(*) AS row_count
    FROM warehouse.fact_payroll
    GROUP BY employee_id, run_id
    HAVING COUNT(*) > 1
""").fetchdf()

dq04 = con_dq.execute("""
    SELECT f.payroll_item_id, f.employee_id, f.run_id
    FROM warehouse.fact_payroll f
    LEFT JOIN warehouse.dim_employee d
        ON f.employee_id = d.employee_id
    WHERE d.employee_id IS NULL
""").fetchdf()

initial_results = {
    "DQ-C01": dq01,
    "DQ-C02": dq02,
    "DQ-C03": dq03,
    "DQ-C04": dq04,
}

for check_id, bad_rows in initial_results.items():
    print(f"\n{check_id}: {len(bad_rows)} failing result row(s)")
    display(bad_rows.head(10))


In [ ]:

# Investigate salary-related ledger postings
ledger_sign_check = con_dq.execute("""
    SELECT
        account_code,
        source_type,
        COUNT(*) AS line_count,
        ROUND(SUM(debit), 2) AS total_debit,
        ROUND(SUM(credit), 2) AS total_credit,
        ROUND(SUM(debit - credit), 2) AS debit_minus_credit,
        ROUND(SUM(signed_amount_debit_positive), 2)
            AS signed_amount_total
    FROM warehouse.fact_ledger
    WHERE account_code BETWEEN 5001 AND 5005
    GROUP BY account_code, source_type
    ORDER BY account_code, source_type
""").fetchdf()

display(ledger_sign_check)

print("\nMonthly reconciliation using the signed amount column:")
display(con_dq.execute("""
    WITH payroll_monthly AS (
        SELECT
            DATE_TRUNC('month', period_end_date)::DATE AS month,
            SUM(employer_cost) AS payroll_cost
        FROM warehouse.fact_payroll
        GROUP BY 1
    ),
    ledger_monthly AS (
        SELECT
            DATE_TRUNC('month', entry_date)::DATE AS month,
            SUM(signed_amount_debit_positive) AS ledger_cost
        FROM warehouse.fact_ledger
        WHERE account_code BETWEEN 5001 AND 5005
        GROUP BY 1
    )
    SELECT
        COALESCE(p.month, l.month) AS month,
        p.payroll_cost,
        l.ledger_cost,
        p.payroll_cost - l.ledger_cost AS difference
    FROM payroll_monthly p
    FULL OUTER JOIN ledger_monthly l USING (month)
    WHERE p.month IS NULL
       OR l.month IS NULL
       OR ABS(p.payroll_cost - l.ledger_cost) > 0.01
    ORDER BY month
""").fetchdf())


In [ ]:

# Inspect payroll and salary-related ledger totals by month and account
display(con_dq.execute("""
    WITH payroll_monthly AS (
        SELECT
            DATE_TRUNC('month', period_end_date)::DATE AS month,
            SUM(employer_cost) AS payroll_cost
        FROM warehouse.fact_payroll
        GROUP BY 1
    ),
    ledger_monthly AS (
        SELECT
            DATE_TRUNC('month', entry_date)::DATE AS month,
            account_code,
            SUM(signed_amount_debit_positive) AS account_cost
        FROM warehouse.fact_ledger
        WHERE account_code BETWEEN 5001 AND 5005
        GROUP BY 1, 2
    )
    SELECT
        l.month,
        l.account_code,
        ROUND(l.account_cost, 2) AS account_cost,
        ROUND(p.payroll_cost, 2) AS payroll_cost
    FROM ledger_monthly l
    LEFT JOIN payroll_monthly p USING (month)
    WHERE l.month IN (
        DATE '2025-05-01',
        DATE '2025-12-01',
        DATE '2026-02-01',
        DATE '2026-04-01',
        DATE '2026-06-01'
    )
    ORDER BY l.month, l.account_code
""").fetchdf())


In [ ]:

# PQ-03: Data-quality checks DQ-C05 to DQ-C08

# DQ-C05: Attendance after employee exit
dq05 = con_dq.execute("""
    SELECT
        a.attendance_fact_id,
        a.employee_id,
        a.period_end_date,
        d.exit_date
    FROM warehouse.fact_attendance_monthly a
    JOIN warehouse.dim_employee d
        ON a.employee_id = d.employee_id
       AND a.employee_key = d.employee_key
    WHERE d.exit_date IS NOT NULL
      AND a.period_end_date > LAST_DAY(d.exit_date)
""").fetchdf()

# DQ-C06: Compare unpaid absence days with payroll LOP days
dq06 = con_dq.execute("""
    WITH attendance_monthly AS (
        SELECT
            employee_id,
            DATE_TRUNC('month', period_end_date)::DATE AS month,
            SUM(unpaid_absence_days) AS unpaid_absence_days
        FROM warehouse.fact_attendance_monthly
        GROUP BY 1, 2
    ),
    payroll_monthly AS (
        SELECT
            employee_id,
            DATE_TRUNC('month', period_end_date)::DATE AS month,
            SUM(lop_days) AS lop_days
        FROM warehouse.fact_payroll
        GROUP BY 1, 2
    )
    SELECT
        COALESCE(a.employee_id, p.employee_id) AS employee_id,
        COALESCE(a.month, p.month) AS month,
        a.unpaid_absence_days,
        p.lop_days,
        ABS(
            COALESCE(a.unpaid_absence_days, 0)
            - COALESCE(p.lop_days, 0)
        ) AS difference_days
    FROM attendance_monthly a
    FULL OUTER JOIN payroll_monthly p
        ON a.employee_id = p.employee_id
       AND a.month = p.month
    WHERE ABS(
        COALESCE(a.unpaid_absence_days, 0)
        - COALESCE(p.lop_days, 0)
    ) > 0.5
""").fetchdf()

# DQ-C07: Invoice arithmetic
dq07 = con_dq.execute("""
    SELECT
        invoice_id,
        total_amount,
        amount_paid,
        balance_due
    FROM warehouse.fact_invoice
    WHERE amount_paid > total_amount
       OR balance_due < 0
""").fetchdf()

# DQ-C08: Expense arithmetic
dq08 = con_dq.execute("""
    SELECT
        expense_id,
        amount_excl_gst,
        gst_amount,
        total_amount,
        total_amount - (amount_excl_gst + gst_amount) AS difference
    FROM warehouse.fact_expense
    WHERE ABS(
        total_amount - (amount_excl_gst + gst_amount)
    ) > 0.01
""").fetchdf()

checks_05_08 = {
    "DQ-C05": dq05,
    "DQ-C06": dq06,
    "DQ-C07": dq07,
    "DQ-C08": dq08,
}

for check_id, bad_rows in checks_05_08.items():
    print(f"\n{check_id}: {len(bad_rows)} failing result row(s)")
    display(bad_rows.head(10))


In [ ]:

# PQ-03: Data-quality checks DQ-C09 to DQ-C12

# DQ-C09: Task dates sensible
dq09 = con_dq.execute("""
    SELECT
        task_id,
        status,
        started_date,
        completed_date
    FROM warehouse.fact_task
    WHERE (
        started_date IS NOT NULL
        AND completed_date IS NOT NULL
        AND completed_date < started_date
    )
    OR (
        LOWER(status) = 'done'
        AND completed_date IS NULL
    )
""").fetchdf()

# DQ-C10: Leave dates sensible
dq10 = con_dq.execute("""
    SELECT
        leave_request_id,
        start_date,
        end_date,
        status
    FROM warehouse.fact_leave
    WHERE start_date IS NOT NULL
      AND end_date IS NOT NULL
      AND end_date < start_date
""").fetchdf()

# DQ-C11: AI prediction freshness
dq11 = con_dq.execute("""
    SELECT
        prediction_id,
        as_of_date,
        (SELECT MAX(as_of_date)
         FROM warehouse.fact_attrition_risk) AS latest_as_of_date
    FROM warehouse.fact_attrition_risk
    WHERE as_of_date < (
        SELECT MAX(as_of_date)
        FROM warehouse.fact_attrition_risk
    ) - INTERVAL 7 DAYS
""").fetchdf()

# DQ-C12: Employee dimension history overlaps
dq12 = con_dq.execute("""
    SELECT
        a.employee_id,
        a.employee_key AS earlier_key,
        a.valid_from AS earlier_valid_from,
        a.valid_to AS earlier_valid_to,
        b.employee_key AS later_key,
        b.valid_from AS later_valid_from,
        b.valid_to AS later_valid_to
    FROM warehouse.dim_employee a
    JOIN warehouse.dim_employee b
      ON a.employee_id = b.employee_id
     AND a.employee_key < b.employee_key
     AND a.valid_from <= b.valid_to
     AND b.valid_from <= a.valid_to
""").fetchdf()

checks_09_12 = {
    "DQ-C09": dq09,
    "DQ-C10": dq10,
    "DQ-C11": dq11,
    "DQ-C12": dq12,
}

for check_id, bad_rows in checks_09_12.items():
    print(f"\n{check_id}: {len(bad_rows)} failing result row(s)")
    display(bad_rows.head(10))


In [ ]:

# PQ-03: DQ-C13 and DQ-C14

# DQ-C13: Pipeline freshness
# No etl_run_log table exists, so this check cannot be evaluated yet.
dq13 = pd.DataFrame([{
    "check_id": "DQ-C13",
    "status": "BLOCKED",
    "reason": "etl_run_log table is not available",
    "failing_rows": None
}])

print("DQ-C13 — Pipeline freshness")
display(dq13)

# DQ-C14: Minimum group size
# Evaluate employee counts by department and month for metrics
# with a numeric minimum_group_size.
metric_sizes = con_dq.execute("""
    SELECT
        metric_id,
        metric_name,
        source_table,
        grain,
        minimum_group_size
    FROM source_db.raw.metric_dictionary
    WHERE TRY_CAST(minimum_group_size AS INTEGER) IS NOT NULL
""").fetchdf()

print("Metrics with a defined minimum group size:")
display(metric_sizes)

# Demonstrate the check on employee-level payroll aggregates.
# A department-month with fewer than 5 distinct employees is flagged.
dq14 = con_dq.execute("""
    SELECT
        DATE_TRUNC('month', p.period_end_date)::DATE AS month,
        p.department_id,
        COUNT(DISTINCT p.employee_id) AS employee_count,
        5 AS minimum_group_size
    FROM warehouse.fact_payroll p
    GROUP BY 1, 2
    HAVING COUNT(DISTINCT p.employee_id) < 5
    ORDER BY month, department_id
""").fetchdf()

print("\nDQ-C14 — Small department-month groups:")
print("Failing groups:", len(dq14))
display(dq14.head(20))


In [ ]:

# PQ-03: Consolidate the 14 checks into a summary

check_frames = {
    "DQ-C01": dq01,
    "DQ-C02": dq02,
    "DQ-C03": dq03,
    "DQ-C04": dq04,
    "DQ-C05": dq05,
    "DQ-C06": dq06,
    "DQ-C07": dq07,
    "DQ-C08": dq08,
    "DQ-C09": dq09,
    "DQ-C10": dq10,
    "DQ-C11": dq11,
    "DQ-C12": dq12,
    "DQ-C13": dq13,
    "DQ-C14": dq14,
}

# Map each result to the catalog's severity and name
catalog_lookup = dq_catalog.set_index("check_id")

summary_rows = []

for check_id, bad_rows in check_frames.items():
    catalog_row = catalog_lookup.loc[check_id]

    if check_id == "DQ-C13":
        status = "BLOCKED"
        failing_rows = None
        reason = "etl_run_log table is unavailable"
    else:
        failing_rows = len(bad_rows)
        status = "PASS" if failing_rows == 0 else "FAIL"
        reason = (
            "No failing rows returned"
            if status == "PASS"
            else f"{failing_rows} failing result row(s)"
        )

    summary_rows.append({
        "check_id": check_id,
        "check_name": catalog_row["check_name"],
        "severity": catalog_row["severity"],
        "status": status,
        "failing_rows": failing_rows,
        "reason": reason,
    })

dq_summary = pd.DataFrame(summary_rows)

print("Check summary:")
display(dq_summary)

print("\nStatus counts:")
display(
    dq_summary.groupby("status", dropna=False)
    .size()
    .reset_index(name="check_count")
)

print("\nSeverity and status:")
display(
    dq_summary.groupby(["severity", "status"], dropna=False)
    .size()
    .reset_index(name="check_count")
)


In [ ]:

# PQ-03: Persist the consolidated check summary in the warehouse

from datetime import datetime, timezone

dq_results_to_save = dq_summary.copy()
dq_results_to_save["run_timestamp"] = datetime.now(timezone.utc)
dq_results_to_save["failing_rows"] = (
    dq_results_to_save["failing_rows"].astype("Int64")
)

# Create or replace only the PQ-03 results table in the warehouse
con_dq.register("dq_summary_to_save", dq_results_to_save)

con_dq.execute("""
    CREATE OR REPLACE TABLE warehouse.dq_results AS
    SELECT *
    FROM dq_summary_to_save
""")

con_dq.unregister("dq_summary_to_save")

print("DQ results table saved.")

display(con_dq.execute("""
    SELECT check_id, check_name, severity, status,
           failing_rows, run_timestamp
    FROM warehouse.dq_results
    ORDER BY check_id
""").fetchdf())


In [ ]:

# Inspect departments and the D00 group
print("Department dimension:")
display(con_dq.execute("""
    SELECT *
    FROM warehouse.dim_department
    ORDER BY department_id
""").fetchdf())

print("\nEmployee counts by department:")
display(con_dq.execute("""
    SELECT
        department_id,
        COUNT(DISTINCT employee_id) AS employee_count,
        COUNT(*) AS dimension_rows
    FROM warehouse.dim_employee
    GROUP BY department_id
    ORDER BY department_id
""").fetchdf())

print("\nD00 employee details:")
display(con_dq.execute("""
    SELECT
        employee_id,
        full_name,
        role_title,
        employment_status,
        is_current,
        exit_date
    FROM warehouse.dim_employee
    WHERE department_id = 'D00'
    ORDER BY employee_id
""").fetchdf())


In [ ]:

# DQ-C14: Apply minimum group size and identify suppressed groups

dq14_refined = con_dq.execute("""
    WITH department_month AS (
        SELECT
            DATE_TRUNC('month', p.period_end_date)::DATE AS month,
            p.department_id,
            COUNT(DISTINCT p.employee_id) AS employee_count
        FROM warehouse.fact_payroll p
        GROUP BY 1, 2
    )
    SELECT
        dm.month,
        dm.department_id,
        d.department_name,
        d.is_leadership,
        dm.employee_count,
        CASE
            WHEN dm.employee_count < 5 THEN 'SUPPRESS'
            ELSE 'PASS'
        END AS privacy_status
    FROM department_month dm
    LEFT JOIN warehouse.dim_department d
        ON dm.department_id = d.department_id
    ORDER BY dm.month, dm.department_id
""").fetchdf()

print("Privacy status counts:")
display(
    dq14_refined.groupby("privacy_status")
    .size()
    .reset_index(name="group_count")
)

print("\nGroups requiring suppression:")
display(
    dq14_refined[
        dq14_refined["privacy_status"] == "SUPPRESS"
    ]
)


In [ ]:

# DQ-C06: Classify attendance/payroll mismatches

dq06_review = con_dq.execute("""
    WITH attendance_monthly AS (
        SELECT
            employee_id,
            DATE_TRUNC('month', period_end_date)::DATE AS month,
            SUM(unpaid_absence_days) AS unpaid_absence_days
        FROM warehouse.fact_attendance_monthly
        GROUP BY 1, 2
    ),
    payroll_monthly AS (
        SELECT
            employee_id,
            DATE_TRUNC('month', period_end_date)::DATE AS month,
            SUM(lop_days) AS lop_days
        FROM warehouse.fact_payroll
        GROUP BY 1, 2
    )
    SELECT
        COALESCE(a.employee_id, p.employee_id) AS employee_id,
        COALESCE(a.month, p.month) AS month,
        a.unpaid_absence_days,
        p.lop_days,
        CASE
            WHEN a.employee_id IS NULL THEN 'MISSING_ATTENDANCE'
            WHEN p.employee_id IS NULL THEN 'MISSING_PAYROLL'
            WHEN ABS(a.unpaid_absence_days - p.lop_days) <= 0.5
                THEN 'WITHIN_TOLERANCE'
            ELSE 'DAY_MISMATCH'
        END AS exception_type
    FROM attendance_monthly a
    FULL OUTER JOIN payroll_monthly p
        ON a.employee_id = p.employee_id
       AND a.month = p.month
    WHERE a.employee_id IS NULL
       OR p.employee_id IS NULL
       OR ABS(
            COALESCE(a.unpaid_absence_days, 0)
            - COALESCE(p.lop_days, 0)
       ) > 0.5
    ORDER BY month, employee_id
""").fetchdf()

print("Exception classification:")
display(
    dq06_review.groupby("exception_type")
    .size()
    .reset_index(name="record_count")
)

print("\nSample exceptions:")
display(dq06_review.head(20))


In [ ]:

# Refine DQ-C06 exception counts
dq06_exception_counts = (
    dq06_review.groupby("exception_type")
    .size()
    .to_dict()
)

dq06_total_exceptions = len(dq06_review)

print("DQ-C06 total exceptions:", dq06_total_exceptions)
print("Breakdown:", dq06_exception_counts)

# Update the summary row in memory
dq_summary.loc[dq_summary["check_id"] == "DQ-C06", "failing_rows"] = (
    dq06_total_exceptions
)
dq_summary.loc[dq_summary["check_id"] == "DQ-C06", "status"] = "FAIL"
dq_summary.loc[dq_summary["check_id"] == "DQ-C06", "reason"] = (
    f"{dq06_exception_counts.get('DAY_MISMATCH', 0)} day mismatches; "
    f"{dq06_exception_counts.get('MISSING_ATTENDANCE', 0)} missing attendance; "
    f"{dq06_exception_counts.get('MISSING_PAYROLL', 0)} missing payroll"
)

display(
    dq_summary[dq_summary["check_id"] == "DQ-C06"]
)

print("\nException breakdown for the report:")
display(
    pd.DataFrame([
        {"exception_type": key, "record_count": value}
        for key, value in dq06_exception_counts.items()
    ])
)


In [ ]:

# Persist the revised PQ-03 summary
dq_results_to_save = dq_summary.copy()
dq_results_to_save["run_timestamp"] = datetime.now(timezone.utc)
dq_results_to_save["failing_rows"] = (
    dq_results_to_save["failing_rows"].astype("Int64")
)

con_dq.register("dq_summary_to_save", dq_results_to_save)

con_dq.execute("""
    CREATE OR REPLACE TABLE warehouse.dq_results AS
    SELECT *
    FROM dq_summary_to_save
""")

con_dq.unregister("dq_summary_to_save")

print("Updated dq_results table:")
display(con_dq.execute("""
    SELECT check_id, status, failing_rows, reason
    FROM warehouse.dq_results
    ORDER BY check_id
""").fetchdf())


In [ ]:

# Inspect planted-issues file structure only
issues_path = (
    PROJECT_ROOT
    / "04_Validation_Vault"
    / "unlock_PQ-03_dq_known_issues.csv"
)

issues_metadata = pd.read_csv(issues_path, nrows=0)
issues_count = sum(1 for _ in open(issues_path, encoding="utf-8")) - 1

print("File exists:", issues_path.exists())
print("Number of data rows:", issues_count)
print("Column names:", issues_metadata.columns.tolist())


In [ ]:
issues_path = (
    PROJECT_ROOT
    / "04_Validation_Vault"
    / "unlock_PQ-03_dq_known_issues.csv"
)

planted_issues = pd.read_csv(issues_path)

display(planted_issues)

In [ ]:
# Separate the two planted task issues
task_issues = planted_issues[
    planted_issues["table_name"].eq("fact_task")
]

display(
    task_issues[
        ["issue_id", "issue_type", "affected_record_ids",
         "check_that_should_catch_it"]
    ]
)

# Inspect the task exceptions from our existing DQ-C09 check
display(dq09_review)

In [ ]:
# Fetch the task records that violate either rule
task_review = con_dq.execute("""
    SELECT
        task_id,
        status,
        started_date,
        completed_date,
        CASE
            WHEN completed_date < started_date
                THEN 'COMPLETED_BEFORE_STARTED'
            WHEN LOWER(TRIM(status)) = 'done'
                 AND completed_date IS NULL
                THEN 'DONE_WITHOUT_DATE'
        END AS issue_type
    FROM warehouse.fact_task
    WHERE
        completed_date < started_date
        OR (
            LOWER(TRIM(status)) = 'done'
            AND completed_date IS NULL
        )
    ORDER BY issue_type, task_id
""").fetchdf()

print("Total task exceptions:", len(task_review))
display(task_review)

In [ ]:
# Compare planted task IDs against the detected task IDs
task_issues = planted_issues[
    planted_issues["table_name"].eq("fact_task")
]

task_comparison = []

for _, issue in task_issues.iterrows():
    expected_ids = {
        item.strip()
        for item in str(issue["affected_record_ids"]).split(",")
    }

    detected_ids = set(
        task_review.loc[
            task_review["issue_type"].eq(issue["issue_type"]),
            "task_id"
        ].astype(str)
    )

    matched = expected_ids & detected_ids
    missed = expected_ids - detected_ids

    task_comparison.append({
        "issue_type": issue["issue_type"],
        "expected_count": len(expected_ids),
        "detected_count": len(matched),
        "missed_count": len(missed),
        "missed_ids": ", ".join(sorted(missed)) if missed else "None",
        "status": "PASS" if not missed else "FAIL"
    })

task_comparison_df = pd.DataFrame(task_comparison)
display(task_comparison_df)

In [ ]:
print("Data-quality check summary:")
display(dq_summary)

print("\nTask planted-issue comparison:")
display(task_comparison_df)

print("\nAvailable validation DataFrames:")
print([
    name for name in globals()
    if name.startswith("dq") and
    isinstance(globals()[name], pd.DataFrame)
])

In [ ]:
# Map planted issues to the checks that detect them.
# Use the record ID columns returned by each check.

check_id_columns = {
    "DQ-01": ("dq05", "attendance_fact_id"),
    "DQ-02": ("dq03", "payroll_item_id"),
    "DQ-03": ("dq04", "payroll_item_id"),
    "DQ-04": ("dq01", "posting_period"),
    "DQ-05": ("dq07", "invoice_id"),
    "DQ-06": ("task_review", "task_id"),
    "DQ-07": ("task_review", "task_id"),
    "DQ-08": ("dq10", "leave_request_id"),
    "DQ-09": ("dq11", "prediction_id"),
    "DQ-10": ("dq08", "expense_id"),
    "DQ-11": ("dq12", "employee_key"),
}

print("Available DataFrame columns:")
for issue_id, (var_name, id_col) in check_id_columns.items():
    df = globals().get(var_name)
    print(
        issue_id,
        "| DataFrame:", var_name,
        "| expected ID column:", id_col,
        "| available:", isinstance(df, pd.DataFrame),
        "| columns:", list(df.columns) if isinstance(df, pd.DataFrame) else "N/A"
    )

In [ ]:
# Exact planted-issue coverage audit
coverage_map = {
    "DQ-01": ("dq05", "attendance_fact_id"),
    "DQ-02": ("dq03", None),  # Verify individual payroll IDs directly below
    "DQ-03": ("dq04", "payroll_item_id"),
    "DQ-04": ("dq01", None),  # Monthly mismatch; verify June separately
    "DQ-05": ("dq07", "invoice_id"),
    "DQ-06": ("task_review", "task_id"),
    "DQ-07": ("task_review", "task_id"),
    "DQ-08": ("dq10", "leave_request_id"),
    "DQ-09": ("dq11", "prediction_id"),
    "DQ-10": ("dq08", "expense_id"),
    "DQ-11": ("dq12", None),  # Check either version key
}

coverage_rows = []

for _, issue in planted_issues.iterrows():
    issue_id = str(issue["issue_id"])
    expected_ids = {
        value.strip()
        for value in str(issue["affected_record_ids"]).split(",")
        if value.strip()
    }

    if issue_id == "DQ-01":
        detected_ids = set(dq05["attendance_fact_id"].astype(str))

    elif issue_id == "DQ-02":
        duplicate_details = con_dq.execute("""
            SELECT payroll_item_id
            FROM warehouse.fact_payroll
            WHERE (employee_id, run_id) IN (
                SELECT employee_id, run_id
                FROM warehouse.fact_payroll
                GROUP BY employee_id, run_id
                HAVING COUNT(*) > 1
            )
        """).fetchdf()
        detected_ids = set(duplicate_details["payroll_item_id"].astype(str))

    elif issue_id == "DQ-03":
        detected_ids = set(dq04["payroll_item_id"].astype(str))

    elif issue_id == "DQ-04":
        june = dq01[
            dq01["month"].astype(str).str.startswith("2026-06")
        ]
        detected_ids = expected_ids if not june.empty and abs(
            float(june.iloc[0]["difference"]) - 18450.0
        ) < 0.01 else set()

    elif issue_id in ("DQ-06", "DQ-07"):
        issue_type = (
            "COMPLETED_BEFORE_STARTED"
            if issue_id == "DQ-06"
            else "DONE_WITHOUT_DATE"
        )
        detected_ids = set(
            task_review.loc[
                task_review["issue_type"].eq(issue_type), "task_id"
            ].astype(str)
        )

    elif issue_id == "DQ-11":
        overlap_keys = set(dq12["earlier_key"].astype(str)) | set(
            dq12["later_key"].astype(str)
        )
        detected_ids = expected_ids & overlap_keys

    else:
        df_name, id_col = coverage_map[issue_id]
        df = globals()[df_name]
        detected_ids = (
            set(df[id_col].astype(str))
            if id_col is not None and id_col in df.columns
            else set()
        )

    matched = expected_ids & detected_ids
    missed = expected_ids - detected_ids

    coverage_rows.append({
        "issue_id": issue_id,
        "issue_type": issue["issue_type"],
        "expected_count": len(expected_ids),
        "detected_count": len(matched),
        "missed_count": len(missed),
        "missed_ids": ", ".join(sorted(missed)) if missed else "None",
        "status": "PASS" if not missed else "MISS"
    })

coverage_report = pd.DataFrame(coverage_rows)
display(coverage_report)

In [ ]:
# Verify the two planted journal lines
ledger_issue_lines = con_dq.execute("""
    SELECT
        line_id,
        entry_id,
        entry_date,
        account_code,
        debit,
        credit,
        signed_amount_debit_positive
    FROM warehouse.fact_ledger
    WHERE line_id IN ('JL-005614', 'JL-005643')
       OR entry_id = 'JE-002094'
    ORDER BY entry_id, line_id
""").fetchdf()

print("Matching ledger records:", len(ledger_issue_lines))
display(ledger_issue_lines)

In [ ]:
# Confirm the exact planted journal lines
planted_line_ids = {"JL-005614", "JL-005643"}

found_line_ids = set(
    ledger_issue_lines["line_id"].astype(str)
)

print("Planted lines found:", planted_line_ids & found_line_ids)
print("Planted lines missing:", planted_line_ids - found_line_ids)

# Confirm the June 2026 payroll-ledger difference
june_result = dq01[
    dq01["month"].astype(str).str.startswith("2026-06")
]

display(june_result)

if not june_result.empty:
    difference = float(june_result.iloc[0]["difference"])
    print(f"June difference: ₹{difference:,.2f}")
    print(
        "Reconciliation result:",
        "MATCH" if abs(difference) <= 0.01 else "MISMATCH"
    )

In [ ]:
# Correct DQ-04: the monthly reconciliation detected the planted mismatch
coverage_report.loc[
    coverage_report["issue_id"].astype(str).eq("DQ-04"),
    ["detected_count", "missed_count", "missed_ids", "status"]
] = [2, 0, "None", "PASS"]

print("Updated planted-issue coverage report:")
display(coverage_report)

print("\nCoverage summary:")
print("Issue categories:", len(coverage_report))
print(
    "Categories fully detected:",
    int(coverage_report["status"].eq("PASS").sum())
)
print(
    "Categories with misses:",
    int(coverage_report["status"].ne("PASS").sum())
)

In [ ]:
coverage_path = (
    PROJECT_ROOT
    / "04_Validation_Vault"
    / "PQ-03_planted_issue_coverage.csv"
)

coverage_report.to_csv(coverage_path, index=False)

print("Coverage report saved:", coverage_path)
print("File exists:", coverage_path.exists())
print("Issue categories:", len(coverage_report))
print(
    "Fully detected:",
    int(coverage_report["status"].eq("PASS").sum())
)

In [ ]:
# Find tables whose names may contain ETL or load metadata
tables_df = con_dq.execute("SHOW ALL TABLES").fetchdf()

metadata_candidates = tables_df[
    tables_df["table_name"].astype(str).str.contains(
        "etl|load|run|pipeline|batch", case=False, regex=True
    )
]

display(metadata_candidates)

print("\nCurrent DQ-C13 result:")
display(dq_summary[dq_summary["check_id"].eq("DQ-C13")])

In [ ]:
tables_df = con_dq.execute("SHOW ALL TABLES").fetchdf()

print("Available columns:", tables_df.columns.tolist())
display(tables_df.head(10))

In [ ]:
# Search all database schemas for ETL or pipeline-related tables
metadata_candidates = tables_df[
    tables_df["name"].astype(str).str.contains(
        "etl|load|run|pipeline|batch", case=False, regex=True
    )
]

display(
    metadata_candidates[
        ["database", "schema", "name", "column_names"]
    ]
)

print("\nCurrent DQ-C13 result:")
display(
    dq_summary.loc[dq_summary["check_id"].eq("DQ-C13")]
)

In [ ]:
metric_dictionary = con_dq.execute("""
    SELECT
        metric_id,
        metric_name,
        grain,
        unit,
        minimum_group_size
    FROM source_db.raw.metric_dictionary
    ORDER BY metric_id
""").fetchdf()

print("Metrics in catalog:", len(metric_dictionary))
print("Metrics with a defined minimum group size:",
      metric_dictionary["minimum_group_size"].notna().sum())

display(
    metric_dictionary[
        metric_dictionary["minimum_group_size"].notna()
    ]
)

In [ ]:
import pandas as pd

metric_thresholds = metric_dictionary.copy()

metric_thresholds["minimum_group_size_numeric"] = (
    metric_thresholds["minimum_group_size"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
)

metric_thresholds["minimum_group_size_numeric"] = pd.to_numeric(
    metric_thresholds["minimum_group_size_numeric"],
    errors="coerce"
).astype("Int64")

print("Metrics with numeric thresholds:",
      metric_thresholds["minimum_group_size_numeric"].notna().sum())

print("Metrics without numeric thresholds:",
      metric_thresholds["minimum_group_size_numeric"].isna().sum())

display(
    metric_thresholds[
        [
            "metric_id",
            "metric_name",
            "grain",
            "minimum_group_size",
            "minimum_group_size_numeric"
        ]
    ]
)

In [ ]:
def apply_minimum_group_suppression(
    df,
    group_size_col,
    threshold,
    output_col="release_status"
):
    """
    Mark groups as RELEASE or SUPPRESS based on a minimum size.
    Does not remove rows or alter the original metric values.
    """
    if threshold is None or pd.isna(threshold):
        raise ValueError(
            "Minimum group size is undefined; do not infer a threshold."
        )

    result = df.copy()
    result[output_col] = result[group_size_col].apply(
        lambda n: (
            "SUPPRESS"
            if pd.isna(n) or n < int(threshold)
            else "RELEASE"
        )
    )
    return result


# Test using the existing DQ-C14 department-month results.
# This uses the threshold of 5 for the headcount-style demonstration.
dq14_test = apply_minimum_group_suppression(
    dq14_refined,
    group_size_col="employee_count",
    threshold=5
)

print("Release-status counts:")
display(dq14_test["release_status"].value_counts(dropna=False).to_frame())

print("\nSample rows:")
display(dq14_test.head(10))

In [ ]:
print("DQ-C14 result columns:")
print(dq14_refined.columns.tolist())

print("\nThreshold catalog columns:")
print(metric_thresholds.columns.tolist())

print("\nMetrics with defined thresholds:")
display(
    metric_thresholds.loc[
        metric_thresholds["minimum_group_size_numeric"].notna(),
        [
            "metric_id",
            "metric_name",
            "grain",
            "minimum_group_size_numeric"
        ]
    ]
)

In [ ]:
dq_summary.loc[
    dq_summary["check_id"].eq("DQ-C14"),
    "reason"
] = (
    "PARTIAL: department-month headcount suppression tested "
    "(180 RELEASE, 18 SUPPRESS) using threshold 5; "
    "catalog has 19 metrics with defined thresholds across "
    "multiple grains. Metric-specific aggregation and suppression "
    "rules remain to be implemented."
)

display(
    dq_summary.loc[
        dq_summary["check_id"].eq("DQ-C14"),
        ["check_id", "check_name", "severity", "status",
         "failing_rows", "reason"]
    ]
)

In [ ]:
from datetime import datetime, timezone

# Prepare the latest check summary
dq_results_to_save = dq_summary.copy()
dq_results_to_save["run_timestamp"] = datetime.now(timezone.utc)
dq_results_to_save["failing_rows"] = pd.to_numeric(
    dq_results_to_save["failing_rows"],
    errors="coerce"
).astype("Int64")

# Save the current results to the warehouse database
con_dq.register("dq_summary_to_save", dq_results_to_save)

con_dq.execute("""
    CREATE OR REPLACE TABLE warehouse.dq_results AS
    SELECT * FROM dq_summary_to_save
""")

con_dq.unregister("dq_summary_to_save")

# Verify the persisted results
saved_results = con_dq.execute("""
    SELECT
        check_id,
        check_name,
        severity,
        status,
        failing_rows,
        reason,
        run_timestamp
    FROM warehouse.dq_results
    ORDER BY check_id
""").fetchdf()

print("Saved checks:", len(saved_results))
display(saved_results)

In [ ]:
# Verify the saved planted-issue coverage report
coverage_path = (
    PROJECT_ROOT
    / "04_Validation_Vault"
    / "PQ-03_planted_issue_coverage.csv"
)

saved_coverage = pd.read_csv(coverage_path)

print("Coverage file exists:", coverage_path.exists())
print("Issue categories:", len(saved_coverage))
print(
    "Fully detected:",
    int(saved_coverage["status"].eq("PASS").sum())
)
print(
    "Categories with misses:",
    int(saved_coverage["status"].ne("PASS").sum())
)

display(saved_coverage)

In [ ]:
print("Available DQ-related variables:")

for name in sorted(globals()):
    if name.startswith("dq") and not name.startswith("__"):
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")

print("\nCurrent persisted check results:")
display(
    con_dq.execute("""
        SELECT check_id, check_name, status, failing_rows
        FROM warehouse.dq_results
        ORDER BY check_id
    """).fetchdf()
)

In [ ]:
# Confirm that the current database connections and required tables exist
required_tables = [
    "fact_payroll",
    "fact_ledger",
    "dim_employee",
    "fact_attendance_monthly",
    "fact_invoice",
    "fact_expense",
    "fact_task",
    "fact_leave",
    "fact_attrition_risk",
]

available_tables = con_dq.execute("""
    SHOW ALL TABLES
""").fetchdf()

for table in required_tables:
    found = available_tables[
        (available_tables["database"] == "pulseiq_warehouse")
        & (available_tables["schema"] == "warehouse")
        & (available_tables["name"] == table)
    ].shape[0] > 0

    print(f"{table}: {'AVAILABLE' if found else 'MISSING'}")

print("\nSource connection:")
display(
    con_dq.execute("PRAGMA database_list").fetchdf()
)

In [ ]:
def run_sql_check(query, connection=con_dq):
    """
    Execute a data-quality SQL query and return its failing rows.
    The query should return one row per exception.
    """
    result = connection.execute(query).fetchdf()
    return result


# Test the runner using the ledger balance check (DQ-C02)
ledger_balance_query = """
SELECT
    entry_id,
    ROUND(SUM(debit) - SUM(credit), 2) AS balance_difference
FROM warehouse.fact_ledger
GROUP BY entry_id
HAVING ABS(SUM(debit) - SUM(credit)) > 0.005
"""

dq02_rerun = run_sql_check(ledger_balance_query)

print("Check: DQ-C02 — Ledger balanced")
print("Failing entries:", len(dq02_rerun))
display(dq02_rerun.head(10))

In [ ]:
duplicate_payroll_query = """
SELECT
    employee_id,
    run_id,
    COUNT(*) AS row_count
FROM warehouse.fact_payroll
GROUP BY employee_id, run_id
HAVING COUNT(*) > 1
ORDER BY employee_id, run_id
"""

dq03_rerun = run_sql_check(duplicate_payroll_query)

print("Check: DQ-C03 — Unique payroll rows")
print("Duplicate employee/run groups:", len(dq03_rerun))
display(dq03_rerun)

In [ ]:
dq_sql_checks = {
    "DQ-C02": {
        "name": "Ledger balanced",
        "severity": "Critical",
        "query": """
            SELECT entry_id,
                   ROUND(SUM(debit) - SUM(credit), 2)
                       AS balance_difference
            FROM warehouse.fact_ledger
            GROUP BY entry_id
            HAVING ABS(SUM(debit) - SUM(credit)) > 0.005
        """
    },
    "DQ-C03": {
        "name": "Unique payroll rows",
        "severity": "Critical",
        "query": """
            SELECT employee_id, run_id, COUNT(*) AS row_count
            FROM warehouse.fact_payroll
            GROUP BY employee_id, run_id
            HAVING COUNT(*) > 1
        """
    },
    "DQ-C04": {
        "name": "No orphan employees",
        "severity": "Critical",
        "query": """
            SELECT p.payroll_item_id, p.employee_id, p.run_id
            FROM warehouse.fact_payroll p
            LEFT JOIN warehouse.dim_employee e
                ON p.employee_id = e.employee_id
            WHERE e.employee_id IS NULL
        """
    },
    "DQ-C07": {
        "name": "Invoice arithmetic",
        "severity": "Warning",
        "query": """
            SELECT invoice_id, total_amount, amount_paid, balance_due
            FROM warehouse.fact_invoice
            WHERE amount_paid > total_amount
               OR balance_due < 0
        """
    },
    "DQ-C08": {
        "name": "Expense arithmetic",
        "severity": "Warning",
        "query": """
            SELECT expense_id, amount_excl_gst, gst_amount,
                   total_amount,
                   total_amount - (amount_excl_gst + gst_amount)
                       AS difference
            FROM warehouse.fact_expense
            WHERE ABS(
                total_amount - (amount_excl_gst + gst_amount)
            ) > 0.01
        """
    },
    "DQ-C09": {
        "name": "Task dates sensible",
        "severity": "Warning",
        "query": """
            SELECT task_id, status, started_date, completed_date
            FROM warehouse.fact_task
            WHERE completed_date < started_date
               OR (
                   LOWER(TRIM(status)) = 'done'
                   AND completed_date IS NULL
               )
        """
    },
    "DQ-C10": {
        "name": "Leave dates sensible",
        "severity": "Warning",
        "query": """
            SELECT leave_request_id, start_date, end_date, status
            FROM warehouse.fact_leave
            WHERE end_date < start_date
        """
    },
    "DQ-C11": {
        "name": "AI feeds are fresh",
        "severity": "Warning",
        "query": """
            SELECT prediction_id, as_of_date
            FROM warehouse.fact_attrition_risk
            WHERE as_of_date < (
                SELECT MAX(as_of_date) - INTERVAL '7 days'
                FROM warehouse.fact_attrition_risk
            )
        """
    }
}

print("Reusable SQL checks defined:", len(dq_sql_checks))
print("Check IDs:", sorted(dq_sql_checks.keys()))

In [ ]:
# Execute each reusable SQL definition against current warehouse data
dq_rerun_results = {}
dq_rerun_summary = []

for check_id, check in dq_sql_checks.items():
    try:
        exceptions = run_sql_check(check["query"])
        dq_rerun_results[check_id] = exceptions

        dq_rerun_summary.append({
            "check_id": check_id,
            "check_name": check["name"],
            "severity": check["severity"],
            "status": "PASS" if exceptions.empty else "FAIL",
            "failing_rows": len(exceptions),
            "execution_error": None
        })

    except Exception as exc:
        dq_rerun_summary.append({
            "check_id": check_id,
            "check_name": check["name"],
            "severity": check["severity"],
            "status": "ERROR",
            "failing_rows": None,
            "execution_error": str(exc)
        })

dq_rerun_summary_df = pd.DataFrame(dq_rerun_summary)

display(
    dq_rerun_summary_df.sort_values("check_id").reset_index(drop=True)
)

In [ ]:
dq_sql_checks.update({
    "DQ-C05": {
        "name": "No attendance after exit",
        "severity": "Warning",
        "query": """
            SELECT
                a.attendance_fact_id,
                a.employee_id,
                a.period_end_date,
                e.exit_date
            FROM warehouse.fact_attendance_monthly a
            JOIN warehouse.dim_employee e
                ON a.employee_key = e.employee_key
            WHERE e.exit_date IS NOT NULL
              AND a.period_end_date > e.exit_date
        """
    },
    "DQ-C01": {
        "name": "Payroll equals ledger",
        "severity": "Critical",
        "query": """
            WITH payroll_monthly AS (
                SELECT
                    DATE_TRUNC('month', period_end_date)::DATE AS month,
                    SUM(employer_cost) AS payroll_cost
                FROM warehouse.fact_payroll
                GROUP BY 1
            ),
            ledger_monthly AS (
                SELECT
                    DATE_TRUNC('month', entry_date)::DATE AS month,
                    SUM(signed_amount_debit_positive) AS ledger_cost
                FROM warehouse.fact_ledger
                WHERE account_code IN (5001, 5002, 5003, 5004, 5005)
                GROUP BY 1
            )
            SELECT
                COALESCE(p.month, l.month) AS month,
                p.payroll_cost,
                l.ledger_cost,
                COALESCE(p.payroll_cost, 0)
                    - COALESCE(l.ledger_cost, 0) AS difference
            FROM payroll_monthly p
            FULL OUTER JOIN ledger_monthly l USING (month)
            WHERE ABS(
                COALESCE(p.payroll_cost, 0)
                - COALESCE(l.ledger_cost, 0)
            ) > 0.01
        """
    }
})

print("Reusable SQL checks defined:", len(dq_sql_checks))
print("Check IDs:", sorted(dq_sql_checks.keys()))

In [ ]:
dq_sql_checks["DQ-C12"] = {
    "name": "Dimension history does not overlap",
    "severity": "Critical",
    "query": """
        SELECT
            earlier.employee_id,
            earlier.employee_key AS earlier_key,
            earlier.valid_from AS earlier_valid_from,
            earlier.valid_to AS earlier_valid_to,
            later.employee_key AS later_key,
            later.valid_from AS later_valid_from,
            later.valid_to AS later_valid_to
        FROM warehouse.dim_employee earlier
        JOIN warehouse.dim_employee later
          ON earlier.employee_id = later.employee_id
         AND earlier.employee_key < later.employee_key
         AND earlier.valid_from < COALESCE(
                later.valid_to, DATE '9999-12-31'
             )
         AND later.valid_from < COALESCE(
                earlier.valid_to, DATE '9999-12-31'
             )
    """
}

print("Reusable SQL checks defined:", len(dq_sql_checks))
print("DQ-C12 definition added:", "DQ-C12" in dq_sql_checks)

In [ ]:
dq_sql_checks["DQ-C06"] = {
    "name": "Unpaid leave deducted",
    "severity": "Warning",
    "query": """
        WITH attendance AS (
            SELECT
                employee_id,
                period_end_date,
                SUM(unpaid_absence_days) AS unpaid_days
            FROM warehouse.fact_attendance_monthly
            GROUP BY employee_id, period_end_date
        ),
        payroll AS (
            SELECT
                employee_id,
                period_end_date,
                SUM(lop_days) AS lop_days
            FROM warehouse.fact_payroll
            GROUP BY employee_id, period_end_date
        )
        SELECT
            COALESCE(a.employee_id, p.employee_id) AS employee_id,
            COALESCE(a.period_end_date, p.period_end_date)
                AS period_end_date,
            a.unpaid_days,
            p.lop_days,
            CASE
                WHEN a.employee_id IS NULL THEN 'MISSING_ATTENDANCE'
                WHEN p.employee_id IS NULL THEN 'MISSING_PAYROLL'
                WHEN ABS(a.unpaid_days - p.lop_days) > 0.5
                    THEN 'DAY_MISMATCH'
            END AS issue_type
        FROM attendance a
        FULL OUTER JOIN payroll p
          ON a.employee_id = p.employee_id
         AND a.period_end_date = p.period_end_date
        WHERE a.employee_id IS NULL
           OR p.employee_id IS NULL
           OR ABS(a.unpaid_days - p.lop_days) > 0.5
    """
}

print("Reusable SQL checks defined:", len(dq_sql_checks))
print("Check IDs:", sorted(dq_sql_checks.keys()))

In [ ]:
dq_rerun_results = {}
dq_rerun_summary = []

for check_id, check in sorted(dq_sql_checks.items()):
    try:
        exceptions = run_sql_check(check["query"])
        dq_rerun_results[check_id] = exceptions

        dq_rerun_summary.append({
            "check_id": check_id,
            "check_name": check["name"],
            "severity": check["severity"],
            "status": "PASS" if exceptions.empty else "FAIL",
            "failing_rows": len(exceptions),
            "execution_error": None
        })

    except Exception as exc:
        dq_rerun_summary.append({
            "check_id": check_id,
            "check_name": check["name"],
            "severity": check["severity"],
            "status": "ERROR",
            "failing_rows": None,
            "execution_error": str(exc)
        })

dq_rerun_summary_df = pd.DataFrame(dq_rerun_summary)

display(
    dq_rerun_summary_df.sort_values("check_id").reset_index(drop=True)
)

In [ ]:
dq05_rerun = dq_rerun_results["DQ-C05"]

print("Total attendance-after-exit rows:", len(dq05_rerun))

planted_attendance_ids = {
    "FA-X0001",
    "FA-X0002",
    "FA-X0003",
    "FA-X0004"
}

detected_attendance_ids = set(
    dq05_rerun["attendance_fact_id"].astype(str)
)

print(
    "Planted records detected:",
    len(planted_attendance_ids & detected_attendance_ids)
)
print(
    "Planted records missing:",
    sorted(planted_attendance_ids - detected_attendance_ids)
)

print("\nSample exceptions:")
display(dq05_rerun.head(15))

In [ ]:
dq05_review = dq_rerun_results["DQ-C05"].copy()

dq05_review["days_after_exit"] = (
    pd.to_datetime(dq05_review["period_end_date"])
    - pd.to_datetime(dq05_review["exit_date"])
).dt.days

dq05_review["planted_issue"] = dq05_review[
    "attendance_fact_id"
].isin(planted_attendance_ids)

print("Total flagged:", len(dq05_review))
print("Planted exceptions:", int(dq05_review["planted_issue"].sum()))
print("Other flagged records:", int((~dq05_review["planted_issue"]).sum()))

display(
    dq05_review[
        [
            "attendance_fact_id",
            "employee_id",
            "period_end_date",
            "exit_date",
            "days_after_exit",
            "planted_issue"
        ]
    ].sort_values(
        ["planted_issue", "days_after_exit"],
        ascending=[False, True]
    ).head(20)
)

In [ ]:
print("Attendance table columns:")
display(
    con_dq.execute("""
        DESCRIBE warehouse.fact_attendance_monthly
    """).fetchdf()
)

print("\nDQ-C05 catalog definition:")
display(
    dq_catalog[
        dq_catalog.astype(str)
        .apply(
            lambda col: col.str.contains("DQ-C05", regex=False).any()
        )
        .any(axis=1)
    ]
)

In [ ]:
print("dq_catalog type:", type(dq_catalog))
print("\nCatalog preview:")
display(dq_catalog.head(5))

print("\nSearch catalog for DQ-C05:")
print(
    dq_catalog[
        dq_catalog.astype(str).str.contains(
            "DQ-C05", case=False, regex=False, na=False
        )
    ]
)

In [ ]:
dq_sql_checks["DQ-C05"] = """
SELECT
    a.attendance_fact_id,
    a.employee_id,
    a.period_end_date,
    d.exit_date
FROM warehouse.fact_attendance_monthly a
JOIN warehouse.dim_employee d
    ON a.employee_id = d.employee_id
   AND d.is_current = 'Y'
WHERE d.exit_date IS NOT NULL
  AND DATE_TRUNC('month', a.period_end_date)
      > DATE_TRUNC('month', d.exit_date)
"""

dq05_corrected = run_sql_check(dq_sql_checks["DQ-C05"])

planted_attendance_ids = {
    "FA-X0001", "FA-X0002", "FA-X0003", "FA-X0004"
}

detected_ids = set(dq05_corrected["attendance_fact_id"])
planted_detected = detected_ids.intersection(planted_attendance_ids)

print("Corrected DQ-C05 exceptions:", len(dq05_corrected))
print("Planted exceptions detected:", len(planted_detected))
print("Planted exceptions missed:", planted_attendance_ids - detected_ids)

display(dq05_corrected.head(10))

In [ ]:
dq_rerun_results = {}
dq_rerun_summary = []

for check_id, query in dq_sql_checks.items():
    try:
        result = run_sql_check(query)
        dq_rerun_results[check_id] = result

        dq_rerun_summary.append({
            "check_id": check_id,
            "status": "PASS" if result.empty else "FAIL",
            "failing_rows": len(result),
            "error": None
        })

    except Exception as e:
        dq_rerun_summary.append({
            "check_id": check_id,
            "status": "ERROR",
            "failing_rows": None,
            "error": str(e)
        })

dq_rerun_summary_df = pd.DataFrame(dq_rerun_summary)

display(
    dq_rerun_summary_df.sort_values("check_id").reset_index(drop=True)
)

print("\nTotal SQL checks executed:", len(dq_sql_checks))
print("Checks passed:", (dq_rerun_summary_df["status"] == "PASS").sum())
print("Checks failed:", (dq_rerun_summary_df["status"] == "FAIL").sum())
print("Checks with errors:", (dq_rerun_summary_df["status"] == "ERROR").sum())

In [ ]:
import inspect

print("run_sql_check definition:")
print(inspect.getsource(run_sql_check))

print("\nType of connection:", type(con_dq))
print("Number of registered checks:", len(dq_sql_checks))

print("\nDQ-C01 query type:", type(dq_sql_checks["DQ-C01"]))
print("DQ-C01 query preview:")
print(repr(dq_sql_checks["DQ-C01"])[:500])

In [ ]:
def run_sql_check(check, connection=con_dq):
    """
    Execute a data-quality SQL check.
    Accepts either a SQL string or a dictionary containing 'query'.
    Returns one row per exception.
    """
    if isinstance(check, dict):
        query = check["query"]
    else:
        query = check

    if not isinstance(query, str):
        raise TypeError(
            f"Expected SQL string, received {type(query).__name__}"
        )

    return connection.execute(query).fetchdf()


print("SQL runner updated successfully.")

# Test one metadata-based check
test_result = run_sql_check(dq_sql_checks["DQ-C01"])
print("DQ-C01 execution successful.")
print("Exception rows:", len(test_result))
display(test_result.head())

In [ ]:
dq_rerun_results = {}
dq_rerun_summary = []

for check_id, check in dq_sql_checks.items():
    try:
        result = run_sql_check(check)
        dq_rerun_results[check_id] = result

        dq_rerun_summary.append({
            "check_id": check_id,
            "status": "PASS" if result.empty else "FAIL",
            "failing_rows": len(result),
            "error": None
        })

    except Exception as e:
        dq_rerun_summary.append({
            "check_id": check_id,
            "status": "ERROR",
            "failing_rows": None,
            "error": str(e)
        })

dq_rerun_summary_df = pd.DataFrame(dq_rerun_summary)

display(
    dq_rerun_summary_df.sort_values("check_id").reset_index(drop=True)
)

print("\nTotal checks:", len(dq_sql_checks))
print("PASS:", (dq_rerun_summary_df["status"] == "PASS").sum())
print("FAIL:", (dq_rerun_summary_df["status"] == "FAIL").sum())
print("ERROR:", (dq_rerun_summary_df["status"] == "ERROR").sum())

In [ ]:
# Verify key planted issues against the latest automated results

planted_checks = {
    "DQ-C02": "Ledger balanced",
    "DQ-C03": "Duplicate payroll rows",
    "DQ-C04": "Orphan employees",
    "DQ-C05": "Attendance after exit",
    "DQ-C07": "Invoice arithmetic",
    "DQ-C08": "Expense arithmetic",
    "DQ-C09": "Task date errors",
    "DQ-C10": "Leave date errors",
    "DQ-C11": "Stale predictions",
    "DQ-C12": "Dimension history overlap",
}

for check_id, description in planted_checks.items():
    result = dq_rerun_results[check_id]
    print(f"{check_id} | {description} | Exceptions: {len(result)}")

# Confirm the four attendance planted IDs are all detected
planted_attendance_ids = {
    "FA-X0001", "FA-X0002", "FA-X0003", "FA-X0004"
}

detected_attendance_ids = set(
    dq_rerun_results["DQ-C05"]["attendance_fact_id"]
)

print("\nDQ-C05 planted detection:")
print("Detected:", len(planted_attendance_ids & detected_attendance_ids), "/ 4")
print("Missed:", planted_attendance_ids - detected_attendance_ids)

In [ ]:
display(
    dq_summary[
        dq_summary["check_id"].isin(["DQ-C13", "DQ-C14"])
    ].to_string(index=False)
)

print("\nExisting summary columns:")
print(dq_summary.columns.tolist())

print("\nAvailable warehouse tables matching ETL/run/log:")
display(
    con_dq.execute("""
        SELECT database_name, schema_name, table_name
        FROM duckdb_tables()
        WHERE lower(table_name) LIKE '%etl%'
           OR lower(table_name) LIKE '%run%log%'
        ORDER BY database_name, schema_name, table_name
    """).fetchdf()
)

In [ ]:
# Start with the latest automated SQL results
final_dq_summary = dq_rerun_summary_df.copy()

# Standardize column names and attach catalog metadata
final_dq_summary = final_dq_summary.rename(
    columns={
        "status": "status",
        "failing_rows": "failing_rows"
    }
)

final_dq_summary["check_name"] = final_dq_summary["check_id"].map(
    dq_catalog.set_index("check_id")["check_name"]
)
final_dq_summary["severity"] = final_dq_summary["check_id"].map(
    dq_catalog.set_index("check_id")["severity"]
)
final_dq_summary["reason"] = final_dq_summary["error"]

# Remove the execution-error column after preserving its contents
final_dq_summary = final_dq_summary[
    ["check_id", "check_name", "severity", "status", "failing_rows", "reason"]
]

# Add DQ-C13: no load-log table means freshness cannot be verified
final_dq_summary.loc[len(final_dq_summary)] = {
    "check_id": "DQ-C13",
    "check_name": "Pipeline freshness",
    "severity": "Critical",
    "status": "BLOCKED",
    "failing_rows": None,
    "reason": "etl_run_log table is unavailable; freshness cannot be verified."
}

# Add DQ-C14: partial implementation only
final_dq_summary.loc[len(final_dq_summary)] = {
    "check_id": "DQ-C14",
    "check_name": "Minimum group size",
    "severity": "Critical",
    "status": "PARTIAL",
    "failing_rows": 18,
    "reason": (
        "Department-month headcount suppression tested: "
        "180 RELEASE, 18 SUPPRESS using threshold 5. "
        "Metric-specific thresholds and aggregation grains remain incomplete."
    )
}

final_dq_summary = (
    final_dq_summary
    .sort_values("check_id")
    .reset_index(drop=True)
)

display(final_dq_summary)

print("\nTotal catalog checks:", len(final_dq_summary))
print("\nStatus counts:")
display(final_dq_summary["status"].value_counts().rename_axis("status").to_frame("count"))

In [ ]:
# Distinguish successful execution from data-quality findings
final_dq_summary["execution_status"] = "EXECUTED"

final_dq_summary.loc[
    final_dq_summary["check_id"] == "DQ-C13",
    "execution_status"
] = "BLOCKED"

final_dq_summary.loc[
    final_dq_summary["check_id"] == "DQ-C14",
    "execution_status"
] = "PARTIAL"

# Add clear interpretation for the report
final_dq_summary["interpretation"] = final_dq_summary["status"].map({
    "PASS": "Executed; no exceptions found",
    "FAIL": "Executed; data-quality exceptions found",
    "BLOCKED": "Could not evaluate with available data",
    "PARTIAL": "Partially implemented; further validation required",
    "ERROR": "SQL execution error"
})

display(
    final_dq_summary[
        [
            "check_id",
            "check_name",
            "severity",
            "status",
            "execution_status",
            "failing_rows",
            "interpretation"
        ]
    ]
)

print("\nSummary:")
print("Total checks:", len(final_dq_summary))
print("SQL checks executed:", (final_dq_summary["execution_status"] == "EXECUTED").sum())
print("Blocked checks:", (final_dq_summary["execution_status"] == "BLOCKED").sum())
print("Partial checks:", (final_dq_summary["execution_status"] == "PARTIAL").sum())

In [ ]:
# Save the consolidated 14-check summary to DuckDB
con_dq.register("final_dq_summary_temp", final_dq_summary)

con_dq.execute("""
    CREATE OR REPLACE TABLE warehouse.dq_results AS
    SELECT *
    FROM final_dq_summary_temp
""")

con_dq.unregister("final_dq_summary_temp")

print("Saved table: warehouse.dq_results")

# Verify saved record count and statuses
print("\nSaved record count:")
print(
    con_dq.execute("""
        SELECT COUNT(*) AS total_checks
        FROM warehouse.dq_results
    """).fetchdf()
)

print("\nSaved status summary:")
display(
    con_dq.execute("""
        SELECT status, COUNT(*) AS check_count
        FROM warehouse.dq_results
        GROUP BY status
        ORDER BY status
    """).fetchdf()
)

print("\nSaved report preview:")
display(
    con_dq.execute("""
        SELECT check_id, check_name, severity, status,
               execution_status, failing_rows
        FROM warehouse.dq_results
        ORDER BY check_id
    """).fetchdf()
)

In [ ]:
print("Planted-issue coverage report exists:")
print(
    r"C:\Users\Hp\Documents\pulseiq\04_Validation_Vault"
    r"\PQ-03_planted_issue_coverage.csv"
)

coverage_check = pd.read_csv(
    r"C:\Users\Hp\Documents\pulseiq\04_Validation_Vault"
    r"\PQ-03_planted_issue_coverage.csv"
)

print("\nCoverage report shape:", coverage_check.shape)
print("\nCoverage report columns:")
print(coverage_check.columns.tolist())

display(coverage_check)

In [ ]:
from pathlib import Path
import json

notebook_path = Path(
    r"C:\Users\Hp\Documents\pulseiq"
    r"\PQ-03_Data_Quality_Gatekeeper.ipynb"
)

coverage_path = Path(
    r"C:\Users\Hp\Documents\pulseiq\04_Validation_Vault"
    r"\PQ-03_planted_issue_coverage.csv"
)

print("Notebook exists:", notebook_path.exists())
print("Coverage report exists:", coverage_path.exists())

coverage_check = pd.read_csv(coverage_path)

print("Planted issue categories:", len(coverage_check))
print("Categories passing:", (coverage_check["status"] == "PASS").sum())
print("Total missed records:", coverage_check["missed_count"].sum())

saved_checks = con_dq.execute("""
    SELECT COUNT(*) AS total_checks
    FROM warehouse.dq_results
""").fetchone()[0]

print("Saved DQ checks:", saved_checks)

if notebook_path.exists():
    with notebook_path.open("r", encoding="utf-8") as f:
        notebook_json = json.load(f)

    print("Notebook cells:", len(notebook_json["cells"]))
    print("Notebook JSON valid: True")

In [ ]:

# PQ-03: Inspect warehouse tables before implementing automated runs

warehouse_tables = con_dq.execute("""
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema = 'warehouse'
    ORDER BY table_name
""").fetchdf()

display(warehouse_tables)

print("Warehouse table count:", len(warehouse_tables))
print(
    "ETL run log exists:",
    "etl_run_log" in warehouse_tables["table_name"].tolist()
)
print(
    "DQ results table exists:",
    "dq_results" in warehouse_tables["table_name"].tolist()
)


In [ ]:

# PQ-03: Create a persistent ETL run log

con_dq.execute("""
    CREATE TABLE IF NOT EXISTS warehouse.etl_run_log (
        run_id VARCHAR PRIMARY KEY,
        pipeline_name VARCHAR NOT NULL,
        started_at TIMESTAMP NOT NULL,
        completed_at TIMESTAMP,
        run_status VARCHAR NOT NULL,
        records_processed BIGINT DEFAULT 0,
        failed_checks INTEGER DEFAULT 0,
        error_message VARCHAR
    )
""")

print("ETL run log created or already exists.")

display(
    con_dq.execute("""
        DESCRIBE warehouse.etl_run_log
    """).fetchdf()
)


In [ ]:

# PQ-03: Inspect the current quality-check results

print("QUALITY CHECK SUMMARY")
display(
    con_dq.execute("""
        SELECT
            check_id,
            check_name,
            severity,
            status,
            failing_rows
        FROM warehouse.dq_results
        ORDER BY check_id
    """).fetchdf()
)

print("\nETL RUN LOG")
display(
    con_dq.execute("""
        SELECT *
        FROM warehouse.etl_run_log
        ORDER BY started_at DESC
    """).fetchdf()
)


In [ ]:

from datetime import datetime, timezone
from uuid import uuid4
import pandas as pd

# Start a unique quality-gate run
run_id = str(uuid4())
started_at = datetime.now(timezone.utc).replace(tzinfo=None)

con_dq.execute("""
    INSERT INTO warehouse.etl_run_log (
        run_id, pipeline_name, started_at, run_status
    )
    VALUES (?, ?, ?, ?)
""", [run_id, "PQ-03 Data Quality Gatekeeper", started_at, "RUNNING"])

try:
    # Rerun every implemented SQL check
    rerun_results = {}
    rerun_rows = []

    for check_id, check in dq_sql_checks.items():
        result_df = run_sql_check(check, connection=con_dq)
        rerun_results[check_id] = result_df
        rerun_rows.append({
            "check_id": check_id,
            "failing_rows": len(result_df)
        })

    # Refresh the implemented checks in the saved summary.
    # Preserve BLOCKED/PARTIAL checks that do not yet have complete implementations.
    actual_failures = {
        row["check_id"]: row["failing_rows"]
        for row in rerun_rows
    }

    con_dq.register(
        "pq03_rerun_counts",
        pd.DataFrame(rerun_rows)
    )

    con_dq.execute("""
        UPDATE warehouse.dq_results AS saved
        SET
            failing_rows = rerun.failing_rows,
            status = CASE
                WHEN rerun.failing_rows = 0 THEN 'PASS'
                ELSE 'FAIL'
            END
        FROM pq03_rerun_counts AS rerun
        WHERE saved.check_id = rerun.check_id
    """)

    con_dq.unregister("pq03_rerun_counts")

    # Count checks by status; BLOCKED/PARTIAL are not counted as FAIL.
    summary = con_dq.execute("""
        SELECT status, COUNT(*) AS check_count
        FROM warehouse.dq_results
        GROUP BY status
    """).fetchdf()

    failed_checks = int(
        con_dq.execute("""
            SELECT COUNT(*)
            FROM warehouse.dq_results
            WHERE status = 'FAIL'
        """).fetchone()[0]
    )

    run_status = (
        "FAILED"
        if failed_checks > 0
        else "COMPLETED_WITH_WARNINGS"
    )

    completed_at = datetime.now(timezone.utc).replace(tzinfo=None)

    con_dq.execute("""
        UPDATE warehouse.etl_run_log
        SET completed_at = ?,
            run_status = ?,
            records_processed = ?,
            failed_checks = ?
        WHERE run_id = ?
    """, [
        completed_at,
        run_status,
        sum(len(df) for df in rerun_results.values()),
        failed_checks,
        run_id
    ])

    print("Run ID:", run_id)
    print("Run status:", run_status)
    print("Failed checks:", failed_checks)
    print("\nCurrent check-status counts:")
    display(summary)

except Exception as exc:
    con_dq.execute("""
        UPDATE warehouse.etl_run_log
        SET completed_at = ?,
            run_status = 'ERROR',
            error_message = ?
        WHERE run_id = ?
    """, [
        datetime.now(timezone.utc).replace(tzinfo=None),
        str(exc)[:2000],
        run_id
    ])
    raise

print("\nLatest ETL run:")
display(con_dq.execute("""
    SELECT *
    FROM warehouse.etl_run_log
    ORDER BY started_at DESC
    LIMIT 1
""").fetchdf())


In [ ]:

# PQ-03: Audit the latest run against the saved quality results

print("1. LATEST PIPELINE RUN")

display(con_dq.execute("""
    SELECT
        run_id,
        pipeline_name,
        run_status,
        records_processed,
        failed_checks,
        started_at,
        completed_at,
        error_message
    FROM warehouse.etl_run_log
    ORDER BY started_at DESC
    LIMIT 1
""").fetchdf())

print("\n2. CHECKS REQUIRING ATTENTION")

display(con_dq.execute("""
    SELECT
        check_id,
        check_name,
        severity,
        status,
        failing_rows,
        reason
    FROM warehouse.dq_results
    WHERE status IN ('FAIL', 'BLOCKED', 'PARTIAL')
    ORDER BY
        CASE severity
            WHEN 'Critical' THEN 1
            ELSE 2
        END,
        check_id
""").fetchdf())

print("\n3. CRITICAL FAILURE COUNT")

display(con_dq.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE severity = 'Critical'
              AND status = 'FAIL'
        ) AS critical_failed_checks,
        COUNT(*) FILTER (
            WHERE severity = 'Critical'
              AND status = 'BLOCKED'
        ) AS critical_blocked_checks,
        COUNT(*) FILTER (
            WHERE severity = 'Critical'
              AND status = 'PARTIAL'
        ) AS critical_partial_checks
    FROM warehouse.dq_results
""").fetchdf())


In [ ]:

# PQ-03: Inspect freshness evidence and privacy-rule definitions

print("1. ETL RUN LOG HISTORY")

display(con_dq.execute("""
    SELECT
        pipeline_name,
        run_id,
        started_at,
        completed_at,
        run_status,
        records_processed
    FROM warehouse.etl_run_log
    ORDER BY started_at DESC
    LIMIT 20
""").fetchdf())

print("\n2. METRIC DICTIONARY COLUMNS")

display(con_dq.execute("""
    DESCRIBE source_db.raw.metric_dictionary
""").fetchdf())

print("\n3. METRIC PRIVACY THRESHOLDS")

display(con_dq.execute("""
    SELECT *
    FROM source_db.raw.metric_dictionary
    ORDER BY metric_id
""").fetchdf())

print("\n4. DQ-C13 AND DQ-C14 CATALOG RULES")

display(con_dq.execute("""
    SELECT *
    FROM source_db.raw.dq_checks_catalog
    WHERE check_id IN ('DQ-C13', 'DQ-C14')
    ORDER BY check_id
""").fetchdf())


In [ ]:

# PQ-03 / DQ-C13: Latest successful pipeline run within 24 hours

freshness_result = con_dq.execute("""
    WITH latest_success AS (
        SELECT
            MAX(completed_at) AS latest_successful_load
        FROM warehouse.etl_run_log
        WHERE run_status IN ('SUCCESS', 'SUCCEEDED')
          AND completed_at IS NOT NULL
    )
    SELECT
        latest_successful_load,
        CURRENT_TIMESTAMP AS checked_at,
        CASE
            WHEN latest_successful_load IS NULL
                THEN 'BLOCKED'
            WHEN latest_successful_load <
                 CURRENT_TIMESTAMP - INTERVAL '24 hours'
                THEN 'FAIL'
            ELSE 'PASS'
        END AS status,
        CASE
            WHEN latest_successful_load IS NULL
                THEN 'No successful pipeline run is recorded.'
            WHEN latest_successful_load <
                 CURRENT_TIMESTAMP - INTERVAL '24 hours'
                THEN 'Latest successful run is older than 24 hours.'
            ELSE 'Latest successful run is within 24 hours.'
        END AS reason
    FROM latest_success
""").fetchdf()

display(freshness_result)


In [ ]:

# Save the actual DQ-C13 freshness result

freshness_status = freshness_result.loc[0, "status"]
freshness_reason = freshness_result.loc[0, "reason"]

con_dq.execute("""
    UPDATE warehouse.dq_results
    SET
        status = ?,
        failing_rows = NULL,
        reason = ?
    WHERE check_id = 'DQ-C13'
""", [freshness_status, freshness_reason])

print("Updated DQ-C13:")
display(con_dq.execute("""
    SELECT check_id, check_name, severity, status,
           failing_rows, reason
    FROM warehouse.dq_results
    WHERE check_id = 'DQ-C13'
""").fetchdf())


In [ ]:

# PQ-03 / DQ-C14: Inspect columns needed for privacy-safe aggregation

for table_name in [
    "fact_headcount_monthly",
    "fact_payroll",
    "fact_attendance_monthly",
    "fact_task",
    "fact_attrition_risk"
]:
    print(f"\n--- warehouse.{table_name} ---")
    display(
        con_dq.execute(
            f'DESCRIBE warehouse."{table_name}"'
        ).fetchdf()
    )


In [ ]:

# PQ-03 / DQ-C14: Build the metric-specific privacy threshold map

metric_thresholds = con_dq.execute("""
    SELECT
        metric_id,
        metric_name,
        source_table,
        grain,
        minimum_group_size,
        TRY_CAST(
            REGEXP_EXTRACT(minimum_group_size, '[0-9]+') AS INTEGER
        ) AS min_group_size
    FROM source_db.raw.metric_dictionary
    WHERE minimum_group_size IS NOT NULL
      AND TRIM(minimum_group_size) <> ''
      AND LOWER(TRIM(minimum_group_size)) <> 'nan'
    ORDER BY metric_id
""").fetchdf()

print("Metrics with defined thresholds:", len(metric_thresholds))
display(metric_thresholds)

print("\nThreshold distribution:")
display(
    metric_thresholds.groupby("min_group_size")
    .size()
    .reset_index(name="metric_count")
    .sort_values("min_group_size")
)


In [ ]:

# PQ-03 / DQ-C14: M01 Headcount privacy check
# Count distinct employees by department and month-end date.

m01_headcount_privacy = con_dq.execute("""
    WITH month_ends AS (
        SELECT
            month_start_date,
            department_id,
            headcount_end
        FROM warehouse.fact_headcount_monthly
    ),
    employee_counts AS (
        SELECT
            h.month_start_date,
            h.department_id,
            h.headcount_end,
            COUNT(DISTINCT e.employee_id) AS employee_count
        FROM month_ends h
        LEFT JOIN warehouse.dim_employee e
            ON e.department_id = h.department_id
           AND e.hire_date <= (
               h.month_start_date + INTERVAL '1 month' - INTERVAL '1 day'
           )
           AND (
               e.exit_date IS NULL
               OR e.exit_date > (
                   h.month_start_date + INTERVAL '1 month' - INTERVAL '1 day'
               )
           )
        GROUP BY
            h.month_start_date,
            h.department_id,
            h.headcount_end
    )
    SELECT
        month_start_date,
        department_id,
        headcount_end,
        employee_count,
        5 AS min_group_size,
        CASE
            WHEN employee_count >= 5 THEN 'RELEASE'
            ELSE 'SUPPRESS'
        END AS privacy_status
    FROM employee_counts
    ORDER BY month_start_date, department_id
""").fetchdf()

print("M01 department-month groups:", len(m01_headcount_privacy))
display(m01_headcount_privacy.head(20))

print("\nPrivacy summary:")
display(
    m01_headcount_privacy["privacy_status"]
    .value_counts()
    .rename_axis("privacy_status")
    .reset_index(name="group_count")
)


In [ ]:

# PQ-03 / DQ-C14: Reconcile calculated and recorded headcount

m01_headcount_reconciliation = m01_headcount_privacy.copy()

m01_headcount_reconciliation["headcount_difference"] = (
    m01_headcount_reconciliation["employee_count"]
    - m01_headcount_reconciliation["headcount_end"]
)

m01_mismatches = m01_headcount_reconciliation.loc[
    m01_headcount_reconciliation["headcount_difference"] != 0
].copy()

print("Total department-month groups:", len(m01_headcount_reconciliation))
print("Groups with headcount mismatches:", len(m01_mismatches))

display(
    m01_mismatches.sort_values(
        ["month_start_date", "department_id"]
    ).head(25)
)

print("\nMismatch counts by direction:")
display(
    m01_mismatches.assign(
        mismatch_direction=m01_mismatches["headcount_difference"].apply(
            lambda x: "Calculated higher" if x > 0 else "Calculated lower"
        )
    )["mismatch_direction"]
    .value_counts()
    .rename_axis("mismatch_direction")
    .reset_index(name="group_count")
)


In [ ]:

# PQ-03 / DQ-C14: Inspect employee history and department assignments

print("1. EMPLOYEE DIMENSION COLUMNS")
display(
    con_dq.execute("""
        DESCRIBE warehouse.dim_employee
    """).fetchdf()
)

print("\n2. EMPLOYEE VERSION SUMMARY")
display(
    con_dq.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT employee_id) AS distinct_employees,
            COUNT(*) FILTER (WHERE is_current = 'Y') AS current_rows,
            COUNT(*) FILTER (WHERE is_current <> 'Y') AS historical_rows
        FROM warehouse.dim_employee
    """).fetchdf()
)

print("\n3. DEPARTMENT ASSIGNMENTS PER EMPLOYEE")
display(
    con_dq.execute("""
        SELECT
            employee_id,
            COUNT(*) AS dimension_versions,
            COUNT(DISTINCT department_id) AS department_count
        FROM warehouse.dim_employee
        GROUP BY employee_id
        HAVING COUNT(*) > 1
            OR COUNT(DISTINCT department_id) > 1
        ORDER BY department_count DESC, dimension_versions DESC
        LIMIT 20
    """).fetchdf())


In [ ]:

# PQ-03 / DQ-C14: Correct M01 for SCD Type 2 history

m01_scd_reconciliation = con_dq.execute("""
    WITH employee_counts AS (
        SELECT
            h.month_start_date,
            h.department_id,
            h.headcount_end,
            COUNT(DISTINCT e.employee_id) AS employee_count
        FROM warehouse.fact_headcount_monthly h
        LEFT JOIN warehouse.dim_employee e
            ON e.department_id = h.department_id
           AND e.valid_from <= (
               h.month_start_date
               + INTERVAL '1 month'
               - INTERVAL '1 day'
           )
           AND (
               e.valid_to IS NULL
               OR e.valid_to >= (
                   h.month_start_date
                   + INTERVAL '1 month'
                   - INTERVAL '1 day'
               )
           )
           AND e.hire_date <= (
               h.month_start_date
               + INTERVAL '1 month'
               - INTERVAL '1 day'
           )
           AND (
               e.exit_date IS NULL
               OR e.exit_date > (
                   h.month_start_date
                   + INTERVAL '1 month'
                   - INTERVAL '1 day'
               )
           )
        GROUP BY
            h.month_start_date,
            h.department_id,
            h.headcount_end
    )
    SELECT
        *,
        employee_count - headcount_end AS headcount_difference,
        CASE
            WHEN employee_count >= 5 THEN 'RELEASE'
            ELSE 'SUPPRESS'
        END AS privacy_status
    FROM employee_counts
    ORDER BY month_start_date, department_id
""").fetchdf()

m01_scd_mismatches = m01_scd_reconciliation[
    m01_scd_reconciliation["headcount_difference"] != 0
]

print("Total department-month groups:", len(m01_scd_reconciliation))
print("Groups with remaining mismatches:", len(m01_scd_mismatches))
display(m01_scd_mismatches.head(20))

print("\nPrivacy summary:")
display(
    m01_scd_reconciliation["privacy_status"]
    .value_counts()
    .rename_axis("privacy_status")
    .reset_index(name="group_count")
)


In [ ]:

# PQ-03 / DQ-C14: Inspect department-month measures and definitions

print("1. HEADCOUNT FACT SAMPLE")
display(con_dq.execute("""
    SELECT
        month_start_date,
        department_id,
        headcount_start,
        hires,
        exits,
        voluntary_exits,
        headcount_end,
        attrition_rate_monthly_pct
    FROM warehouse.fact_headcount_monthly
    ORDER BY month_start_date, department_id
    LIMIT 15
""").fetchdf())

print("\n2. METRIC DEFINITIONS M01-M06")
display(con_dq.execute("""
    SELECT
        metric_id,
        metric_name,
        formula,
        grain,
        minimum_group_size,
        caveat
    FROM source_db.raw.metric_dictionary
    WHERE metric_id IN ('M01', 'M02', 'M03', 'M04', 'M05', 'M06')
    ORDER BY metric_id
""").fetchdf())


In [ ]:

# PQ-03 / DQ-C14: Privacy assessment for metrics M01-M06

department_month_metrics = con_dq.execute("""
    WITH monthly AS (
        SELECT
            month_start_date,
            department_id,
            headcount_start,
            hires,
            exits,
            voluntary_exits,
            headcount_end,
            attrition_rate_monthly_pct
        FROM warehouse.fact_headcount_monthly
    ),
    rolling AS (
        SELECT
            *,
            SUM(exits) OVER (
                PARTITION BY department_id
                ORDER BY month_start_date
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS exits_12m,
            SUM(voluntary_exits) OVER (
                PARTITION BY department_id
                ORDER BY month_start_date
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS voluntary_exits_12m,
            AVG(headcount_end) OVER (
                PARTITION BY department_id
                ORDER BY month_start_date
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS avg_headcount_12m,
            COUNT(*) OVER (
                PARTITION BY department_id
                ORDER BY month_start_date
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS months_in_window
        FROM monthly
    ),
    metric_groups AS (
        SELECT
            month_start_date,
            department_id,
            'M01' AS metric_id,
            'Headcount' AS metric_name,
            headcount_end::DOUBLE AS metric_value,
            headcount_end AS group_size,
            5 AS min_group_size
        FROM rolling

        UNION ALL

        SELECT month_start_date, department_id,
            'M02', 'Hires', hires::DOUBLE, headcount_end, 5
        FROM rolling

        UNION ALL

        SELECT month_start_date, department_id,
            'M03', 'Exits', exits::DOUBLE, headcount_end, 5
        FROM rolling

        UNION ALL

        SELECT month_start_date, department_id,
            'M04', 'Monthly attrition rate',
            CASE WHEN headcount_start > 0
                 THEN exits * 100.0 / headcount_start
                 ELSE NULL END,
            headcount_start, 10
        FROM rolling

        UNION ALL

        SELECT month_start_date, department_id,
            'M05', 'Rolling 12-month attrition rate',
            CASE WHEN avg_headcount_12m > 0
                       AND months_in_window = 12
                 THEN exits_12m * 100.0 / avg_headcount_12m
                 ELSE NULL END,
            CASE WHEN months_in_window = 12
                 THEN CAST(ROUND(avg_headcount_12m) AS BIGINT)
                 ELSE 0 END,
            10
        FROM rolling

        UNION ALL

        SELECT month_start_date, department_id,
            'M06', 'Rolling 12-month voluntary attrition rate',
            CASE WHEN avg_headcount_12m > 0
                       AND months_in_window = 12
                 THEN voluntary_exits_12m * 100.0 / avg_headcount_12m
                 ELSE NULL END,
            CASE WHEN months_in_window = 12
                 THEN CAST(ROUND(avg_headcount_12m) AS BIGINT)
                 ELSE 0 END,
            10
        FROM rolling
    )
    SELECT
        month_start_date,
        department_id,
        metric_id,
        metric_name,
        metric_value,
        group_size,
        min_group_size,
        CASE
            WHEN group_size >= min_group_size THEN 'RELEASE'
            ELSE 'SUPPRESS'
        END AS privacy_status
    FROM metric_groups
    ORDER BY metric_id, month_start_date, department_id
""").fetchdf()

print("Metric-group rows:", len(department_month_metrics))
display(department_month_metrics.head(25))

print("\nSummary by metric and privacy status:")
display(
    department_month_metrics.groupby(
        ["metric_id", "metric_name", "privacy_status"]
    ).size().reset_index(name="group_count")
)


In [ ]:

# PQ-03 / DQ-C14: Validate hires and exits against employee history

hire_exit_validation = con_dq.execute("""
    WITH month_bounds AS (
        SELECT
            month_start_date,
            department_id,
            hires AS recorded_hires,
            exits AS recorded_exits,
            headcount_end
        FROM warehouse.fact_headcount_monthly
    ),
    employee_events AS (
        SELECT
            h.month_start_date,
            h.department_id,
            h.recorded_hires,
            h.recorded_exits,
            h.headcount_end,

            COUNT(DISTINCT CASE
                WHEN e.hire_date >= h.month_start_date
                 AND e.hire_date < h.month_start_date + INTERVAL '1 month'
                THEN e.employee_id
            END) AS distinct_hires,

            COUNT(DISTINCT CASE
                WHEN e.exit_date >= h.month_start_date
                 AND e.exit_date < h.month_start_date + INTERVAL '1 month'
                THEN e.employee_id
            END) AS distinct_exits

        FROM month_bounds h
        LEFT JOIN warehouse.dim_employee e
            ON e.department_id = h.department_id
           AND e.valid_from < h.month_start_date + INTERVAL '1 month'
           AND (
               e.valid_to IS NULL
               OR e.valid_to >= h.month_start_date
           )
        GROUP BY
            h.month_start_date,
            h.department_id,
            h.recorded_hires,
            h.recorded_exits,
            h.headcount_end
    )
    SELECT
        *,
        distinct_hires - recorded_hires AS hires_difference,
        distinct_exits - recorded_exits AS exits_difference
    FROM employee_events
    WHERE distinct_hires <> recorded_hires
       OR distinct_exits <> recorded_exits
    ORDER BY month_start_date, department_id
""").fetchdf()

print("Department-month groups with event-count differences:",
      len(hire_exit_validation))
display(hire_exit_validation.head(25))


In [ ]:
m02_m03_group_validation = con_dq.execute("""
    WITH month_end AS (
        SELECT
            month_start_date,
            department_id,
            headcount_end,
            COUNT(*) OVER (
                PARTITION BY month_start_date, department_id
            ) AS source_group_rows
        FROM warehouse.fact_headcount_monthly
    ),
    actual_employee_counts AS (
        SELECT
            h.month_start_date,
            h.department_id,
            h.headcount_end,
            COUNT(DISTINCT e.employee_id) AS actual_employee_count
        FROM month_end h
        LEFT JOIN warehouse.dim_employee e
            ON e.department_id = h.department_id
           AND e.valid_from <= (
               h.month_start_date + INTERVAL '1 month' - INTERVAL '1 day'
           )
           AND (
               e.valid_to IS NULL
               OR e.valid_to >= (
                   h.month_start_date + INTERVAL '1 month' - INTERVAL '1 day'
               )
           )
           AND e.hire_date <= (
               h.month_start_date + INTERVAL '1 month' - INTERVAL '1 day'
           )
           AND (
               e.exit_date IS NULL
               OR e.exit_date > (
                   h.month_start_date + INTERVAL '1 month' - INTERVAL '1 day'
               )
           )
        GROUP BY
            h.month_start_date,
            h.department_id,
            h.headcount_end
    )
    SELECT *,
        actual_employee_count - headcount_end AS difference
    FROM actual_employee_counts
    WHERE actual_employee_count <> headcount_end
    ORDER BY month_start_date, department_id
""").fetchdf()

print("M02/M03 group-size mismatches:", len(m02_m03_group_validation))
display(m02_m03_group_validation.head(20))

In [ ]:
m04_group_validation = con_dq.execute("""
    WITH actual_start_counts AS (
        SELECT
            h.month_start_date,
            h.department_id,
            h.headcount_start,
            COUNT(DISTINCT e.employee_id) AS actual_employee_count
        FROM warehouse.fact_headcount_monthly h
        LEFT JOIN warehouse.dim_employee e
            ON e.department_id = h.department_id
           AND e.valid_from <= h.month_start_date
           AND (
               e.valid_to IS NULL
               OR e.valid_to >= h.month_start_date
           )
           AND e.hire_date < h.month_start_date
           AND (
               e.exit_date IS NULL
               OR e.exit_date >= h.month_start_date
           )
        GROUP BY
            h.month_start_date,
            h.department_id,
            h.headcount_start
    )
    SELECT *,
        actual_employee_count - headcount_start AS difference
    FROM actual_start_counts
    WHERE actual_employee_count <> headcount_start
    ORDER BY month_start_date, department_id
""").fetchdf()

print("M04 group-size mismatches:", len(m04_group_validation))
display(m04_group_validation.head(20))

In [ ]:
rolling_group_validation = con_dq.execute("""
    WITH monthly AS (
        SELECT
            month_start_date,
            department_id,
            headcount_end,
            COUNT(*) OVER (
                PARTITION BY department_id
                ORDER BY month_start_date
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS months_in_window,
            AVG(headcount_end) OVER (
                PARTITION BY department_id
                ORDER BY month_start_date
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS avg_headcount_12m
        FROM warehouse.fact_headcount_monthly
    )
    SELECT
        COUNT(*) AS total_department_months,
        SUM(CASE WHEN months_in_window = 12 THEN 1 ELSE 0 END)
            AS complete_12m_windows,
        SUM(CASE WHEN months_in_window < 12 THEN 1 ELSE 0 END)
            AS incomplete_12m_windows,
        SUM(CASE
            WHEN months_in_window = 12
             AND avg_headcount_12m >= 10
            THEN 1 ELSE 0
        END) AS complete_windows_meeting_threshold,
        SUM(CASE
            WHEN months_in_window = 12
             AND avg_headcount_12m < 10
            THEN 1 ELSE 0
        END) AS complete_windows_below_threshold
    FROM monthly
""").fetchdf()

display(rolling_group_validation)

In [ ]:
display(
    department_month_metrics.loc[
        department_month_metrics["metric_id"].isin(["M05", "M06"]),
        [
            "metric_id",
            "month_start_date",
            "department_id",
            "group_size",
            "min_group_size",
            "metric_value",
            "suppression_status"

            
        ]
    ].head(20)
)

In [ ]:
print("Columns in department_month_metrics:")
print(department_month_metrics.columns.tolist())

print("\nM05 and M06 sample records:")
display(
    department_month_metrics.loc[
        department_month_metrics["metric_id"].isin(["M05", "M06"])
    ].head(10)
)

In [ ]:
for metric_id in ["M05", "M06"]:
    metric_df = department_month_metrics[
        department_month_metrics["metric_id"] == metric_id
    ]

    print(f"\n{metric_id} — {metric_df['metric_name'].iloc[0]}")
    print("Total records:", len(metric_df))
    print("\nPrivacy status:")
    print(metric_df["privacy_status"].value_counts(dropna=False))

    print("\nMissing metric values by privacy status:")
    print(
        metric_df.assign(
            metric_missing=metric_df["metric_value"].isna()
        ).groupby(
            ["privacy_status", "metric_missing"],
            dropna=False
        ).size()
    )

    print("\nRecords marked RELEASE with missing values:")
    display(
        metric_df[
            (metric_df["privacy_status"] == "RELEASE")
            & (metric_df["metric_value"].isna())
        ].head(10)
    )

In [ ]:
privacy_summary = (
    department_month_metrics
    .assign(
        metric_missing=department_month_metrics["metric_value"].isna()
    )
    .groupby(
        ["metric_id", "metric_name", "privacy_status"],
        dropna=False
    )
    .agg(
        records=("department_id", "size"),
        missing_values=("metric_missing", "sum"),
        below_threshold=(
            "group_size",
            lambda s: (
                s < department_month_metrics.loc[s.index, "min_group_size"]
            ).sum()
        )
    )
    .reset_index()
    .sort_values(["metric_id", "privacy_status"])
)

display(privacy_summary)

In [ ]:
display(
    con_dq.execute("""
        SELECT *
        FROM warehouse.dq_results
        WHERE check_id = 'DQ-C14'
    """).fetchdf()
)

In [ ]:
validated_metric_ids = ["M01", "M02", "M03", "M04", "M05", "M06"]

validated_metrics = department_month_metrics[
    department_month_metrics["metric_id"].isin(validated_metric_ids)
].copy()

print("Validated metrics:", validated_metrics["metric_id"].nunique())
print("Department-month metric records:", len(validated_metrics))
print("\nPrivacy status counts:")
display(
    validated_metrics.groupby(
        ["metric_id", "privacy_status"]
    ).size().unstack(fill_value=0)
)

print("\nMetric dictionary coverage:")
print("Metrics with defined thresholds:", len(metric_thresholds))
print("Metrics validated:", len(validated_metric_ids))
print("Metrics remaining:", len(metric_thresholds) - len(validated_metric_ids))
print(
    "Remaining metric IDs:",
    sorted(set(metric_thresholds["metric_id"]) - set(validated_metric_ids))
)

In [ ]:
remaining_metric_ids = [
    "M07", "M08", "M09", "M10", "M11", "M12", "M13",
    "M14", "M15", "M22", "M23", "M25", "M27"
]

display(
    metric_thresholds[
        metric_thresholds["metric_id"].isin(remaining_metric_ids)
    ][
        [
            "metric_id",
            "metric_name",
            "source_table",
            "grain",
            "minimum_group_size",
            "min_group_size"
        ]
    ].sort_values("metric_id")
)

In [ ]:
print("Payroll columns:")
print(con_dq.execute("""
    DESCRIBE warehouse.fact_payroll
""").fetchdf()["column_name"].tolist())

print("\nMetric definitions:")
display(
    con_dq.execute("""
        SELECT *
        FROM source_db.raw.metric_dictionary
        WHERE metric_id IN ('M07', 'M08', 'M09', 'M15')
        ORDER BY metric_id
    """).fetchdf()
)

In [ ]:
payroll_metric_group_check = con_dq.execute("""
    WITH employee_month AS (
        SELECT
            DATE_TRUNC('month', period_end_date) AS month_start_date,
            employee_id,
            SUM(gross_pay) AS gross_pay,
            SUM(net_pay) AS net_pay,
            SUM(employer_cost) AS employer_cost,
            SUM(lop_days) AS lop_days
        FROM warehouse.fact_payroll
        GROUP BY
            DATE_TRUNC('month', period_end_date),
            employee_id
    )
    SELECT
        month_start_date,
        COUNT(*) AS employee_month_groups,
        COUNT(DISTINCT employee_id) AS distinct_employees,
        SUM(gross_pay) AS total_gross_pay,
        SUM(net_pay) AS total_net_pay,
        SUM(employer_cost) AS total_employer_cost,
        SUM(lop_days) AS total_lop_days
    FROM employee_month
    GROUP BY month_start_date
    ORDER BY month_start_date
""").fetchdf()

display(payroll_metric_group_check)

In [ ]:
payroll_department_privacy_check = con_dq.execute("""
    SELECT
        DATE_TRUNC('month', period_end_date) AS month_start_date,
        department_id,
        COUNT(DISTINCT employee_id) AS employee_count,
        SUM(gross_pay) AS gross_pay,
        SUM(net_pay) AS net_pay,
        SUM(employer_cost) AS employer_cost,
        SUM(lop_days) AS lop_days,
        CASE
            WHEN COUNT(DISTINCT employee_id) >= 5 THEN 'RELEASE'
            ELSE 'SUPPRESS'
        END AS privacy_status
    FROM warehouse.fact_payroll
    GROUP BY
        DATE_TRUNC('month', period_end_date),
        department_id
    ORDER BY month_start_date, department_id
""").fetchdf()

print("Department-month groups:", len(payroll_department_privacy_check))
print("\nPrivacy status counts:")
print(payroll_department_privacy_check["privacy_status"].value_counts())

print("\nEmployee-count distribution:")
display(
    payroll_department_privacy_check["employee_count"]
    .describe()
    .to_frame("employee_count")
)

print("\nSmallest groups:")
display(
    payroll_department_privacy_check
    .sort_values("employee_count")
    .head(15)
)

In [ ]:
print("Attendance columns:")
print(
    con_dq.execute("""
        DESCRIBE warehouse.fact_attendance_monthly
    """).fetchdf()["column_name"].tolist()
)

print("\nAttendance metric definitions:")
display(
    con_dq.execute("""
        SELECT *
        FROM source_db.raw.metric_dictionary
        WHERE metric_id IN ('M13', 'M14')
        ORDER BY metric_id
    """).fetchdf()
)

In [ ]:
attendance_privacy_metrics = con_dq.execute("""
    WITH department_month AS (
        SELECT
            DATE_TRUNC('month', period_end_date) AS month_start_date,
            department_id,
            COUNT(DISTINCT employee_id) AS group_size,
            SUM(present_days) AS total_present_days,
            SUM(half_days_unpaid) AS total_half_days_unpaid,
            SUM(scheduled_days) AS total_scheduled_days,
            SUM(late_arrivals) AS total_late_arrivals
        FROM warehouse.fact_attendance_monthly
        GROUP BY
            DATE_TRUNC('month', period_end_date),
            department_id
    )
    SELECT
        month_start_date,
        department_id,
        group_size,
        5 AS min_group_size,
        total_present_days,
        total_half_days_unpaid,
        total_scheduled_days,
        total_late_arrivals,
        (
            total_present_days + 0.5 * total_half_days_unpaid
        ) / NULLIF(total_scheduled_days, 0) AS m13_attendance_rate,
        total_late_arrivals / NULLIF(total_present_days, 0)
            AS m14_late_arrival_rate,
        CASE
            WHEN group_size >= 5 THEN 'RELEASE'
            ELSE 'SUPPRESS'
        END AS privacy_status
    FROM department_month
    ORDER BY month_start_date, department_id
""").fetchdf()

print("Department-month groups:", len(attendance_privacy_metrics))
print("\nPrivacy status counts:")
print(attendance_privacy_metrics["privacy_status"].value_counts())

print("\nMissing rate values:")
print(
    attendance_privacy_metrics[
        ["m13_attendance_rate", "m14_late_arrival_rate"]
    ].isna().sum()
)

print("\nSample results:")
display(attendance_privacy_metrics.head(10))

In [ ]:
print("Task columns:")
print(
    con_dq.execute("""
        DESCRIBE warehouse.fact_task
    """).fetchdf()["column_name"].tolist()
)

print("\nTask metric definitions:")
display(
    con_dq.execute("""
        SELECT *
        FROM source_db.raw.metric_dictionary
        WHERE metric_id IN ('M22', 'M23', 'M25')
        ORDER BY metric_id
    """).fetchdf()
)

In [ ]:
print("Task status distribution:")
display(
    con_dq.execute("""
        SELECT
            status,
            COUNT(*) AS task_count
        FROM warehouse.fact_task
        GROUP BY status
        ORDER BY task_count DESC
    """).fetchdf()
)

print("\nTeam-sprint group coverage:")
display(
    con_dq.execute("""
        SELECT
            COUNT(*) AS team_sprint_groups,
            SUM(CASE WHEN task_count >= 5 THEN 1 ELSE 0 END)
                AS groups_meeting_threshold,
            SUM(CASE WHEN task_count < 5 THEN 1 ELSE 0 END)
                AS groups_below_threshold,
            SUM(CASE WHEN task_count = 0 THEN 1 ELSE 0 END)
                AS empty_groups
        FROM (
            SELECT
                team_id,
                sprint_id,
                COUNT(*) AS task_count
            FROM warehouse.fact_task
            GROUP BY team_id, sprint_id
        ) AS grouped_tasks
    """).fetchdf()
)

print("\nTask date-quality summary:")
display(
    con_dq.execute("""
        SELECT
            COUNT(*) AS total_tasks,
            SUM(CASE WHEN status = 'Done' THEN 1 ELSE 0 END)
                AS done_tasks,
            SUM(CASE
                WHEN status = 'Done'
                 AND completed_date IS NULL
                THEN 1 ELSE 0
            END) AS done_without_completion_date,
            SUM(CASE
                WHEN status = 'Done'
                 AND started_date IS NULL
                THEN 1 ELSE 0
            END) AS done_without_start_date,
            SUM(CASE
                WHEN status = 'Done'
                 AND started_date IS NOT NULL
                 AND completed_date IS NOT NULL
                 AND completed_date < started_date
                THEN 1 ELSE 0
            END) AS negative_cycle_time_tasks
        FROM warehouse.fact_task
    """).fetchdf()
)

In [ ]:
task_privacy_metrics = con_dq.execute("""
    WITH team_sprint AS (
        SELECT
            team_id,
            sprint_id,
            COUNT(*) AS total_tasks,
            COUNT(*) FILTER (
                WHERE status <> 'Cancelled'
            ) AS planned_tasks,
            COUNT(*) FILTER (
                WHERE status = 'Done'
            ) AS done_tasks,
            COUNT(*) FILTER (
                WHERE completed_on_time = 'Y'
            ) AS on_time_tasks,
            COUNT(*) FILTER (
                WHERE status = 'Done'
                  AND started_date IS NOT NULL
                  AND completed_date IS NOT NULL
                  AND completed_date >= started_date
            ) AS valid_cycle_time_tasks,
            COUNT(*) FILTER (
                WHERE status = 'Done'
                  AND (
                      started_date IS NULL
                      OR completed_date IS NULL
                      OR completed_date < started_date
                  )
            ) AS invalid_cycle_time_tasks,
            AVG(
                DATE_DIFF('day', started_date, completed_date)
            ) FILTER (
                WHERE status = 'Done'
                  AND started_date IS NOT NULL
                  AND completed_date IS NOT NULL
                  AND completed_date >= started_date
            ) AS avg_cycle_time_days
        FROM warehouse.fact_task
        GROUP BY team_id, sprint_id
    )
    SELECT
        team_id,
        sprint_id,
        total_tasks,
        planned_tasks,
        done_tasks,
        on_time_tasks,
        valid_cycle_time_tasks,
        invalid_cycle_time_tasks,
        CASE
            WHEN planned_tasks > 0
            THEN done_tasks::DOUBLE / planned_tasks
        END AS m22_completion_rate,
        CASE
            WHEN done_tasks > 0
            THEN on_time_tasks::DOUBLE / done_tasks
        END AS m23_on_time_rate,
        avg_cycle_time_days AS m25_avg_cycle_time_days,
        CASE
            WHEN total_tasks >= 5 THEN 'RELEASE'
            ELSE 'SUPPRESS'
        END AS privacy_status
    FROM team_sprint
    ORDER BY team_id, sprint_id
""").fetchdf()

print("Team-sprint groups:", len(task_privacy_metrics))
print("\nPrivacy status:")
print(task_privacy_metrics["privacy_status"].value_counts())

print("\nMissing metric values:")
print(
    task_privacy_metrics[
        ["m22_completion_rate", "m23_on_time_rate", "m25_avg_cycle_time_days"]
    ].isna().sum()
)

print("\nGroups containing invalid cycle-time tasks:")
display(
    task_privacy_metrics[
        task_privacy_metrics["invalid_cycle_time_tasks"] > 0
    ].head(15)
)

print("\nSample metric results:")
display(task_privacy_metrics.head(10))

In [ ]:
print("Warehouse tables containing sprint information:")
display(
    con_dq.execute("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_schema = 'warehouse'
          AND (
              LOWER(table_name) LIKE '%sprint%'
              OR table_name = 'dim_sprint'
          )
        ORDER BY table_name
    """).fetchdf()
)

print("\nSprint dimension columns:")
display(
    con_dq.execute("""
        DESCRIBE warehouse.dim_sprint
    """).fetchdf()
)

print("\nSample sprint records:")
display(
    con_dq.execute("""
        SELECT *
        FROM warehouse.dim_sprint
        LIMIT 10
    """).fetchdf()
)

In [ ]:
m23_flag_validation = con_dq.execute("""
    SELECT
        COUNT(*) AS total_done_tasks,
        COUNT(*) FILTER (
            WHERE t.completed_date IS NULL
        ) AS done_missing_completion_date,
        COUNT(*) FILTER (
            WHERE s.end_date IS NULL
        ) AS tasks_missing_sprint_end_date,
        COUNT(*) FILTER (
            WHERE t.completed_date IS NOT NULL
              AND s.end_date IS NOT NULL
              AND t.completed_on_time = 'Y'
              AND t.completed_date > s.end_date
        ) AS flagged_on_time_but_late,
        COUNT(*) FILTER (
            WHERE t.completed_date IS NOT NULL
              AND s.end_date IS NOT NULL
              AND t.completed_on_time <> 'Y'
              AND t.completed_date <= s.end_date
        ) AS flagged_late_but_on_time
    FROM warehouse.fact_task t
    LEFT JOIN warehouse.dim_sprint s
        ON t.sprint_id = s.sprint_id
    WHERE t.status = 'Done'
""").fetchdf()

display(m23_flag_validation)

In [ ]:
print("Data types:")
display(
    con_dq.execute("""
        DESCRIBE warehouse.fact_task
    """).fetchdf().query(
        "column_name in ['status', 'completed_on_time', 'completed_date']"
    )
)

print("\nActual completed_on_time values:")
display(
    con_dq.execute("""
        SELECT
            status,
            completed_on_time,
            COUNT(*) AS task_count
        FROM warehouse.fact_task
        GROUP BY status, completed_on_time
        ORDER BY status, completed_on_time
    """).fetchdf()
)

print("\nDone-task sample:")
display(
    con_dq.execute("""
        SELECT
            task_id,
            status,
            completed_date,
            completed_on_time
        FROM warehouse.fact_task
        WHERE status = 'Done'
        LIMIT 10
    """).fetchdf()
)

In [ ]:
m23_flag_validation = con_dq.execute("""
    SELECT
        COUNT(*) AS total_done_tasks,

        SUM(CASE
            WHEN completed_date IS NULL THEN 1 ELSE 0
        END) AS done_missing_completion_date,

        SUM(CASE
            WHEN s.end_date IS NULL THEN 1 ELSE 0
        END) AS tasks_missing_sprint_end_date,

        SUM(CASE
            WHEN completed_date IS NOT NULL
             AND s.end_date IS NOT NULL
             AND completed_on_time = 'Y'
             AND completed_date > s.end_date
            THEN 1 ELSE 0
        END) AS flagged_on_time_but_late,

        SUM(CASE
            WHEN completed_date IS NOT NULL
             AND s.end_date IS NOT NULL
             AND completed_on_time = 'N'
             AND completed_date <= s.end_date
            THEN 1 ELSE 0
        END) AS flagged_late_but_on_time

    FROM warehouse.fact_task t
    LEFT JOIN warehouse.dim_sprint s
        ON t.sprint_id = s.sprint_id
    WHERE t.status = 'Done'
""").fetchdf()

display(m23_flag_validation)

In [ ]:
print("1. Done tasks without any join:")
display(
    con_dq.execute("""
        SELECT
            status,
            COUNT(*) AS row_count
        FROM warehouse.fact_task
        WHERE status = 'Done'
        GROUP BY status
    """).fetchdf()
)

print("2. Done tasks after joining sprint dimension:")
display(
    con_dq.execute("""
        SELECT
            COUNT(*) AS joined_row_count,
            COUNT(t.task_id) AS task_id_count,
            COUNT(*) FILTER (WHERE t.status = 'Done') AS done_filter_count
        FROM warehouse.fact_task t
        LEFT JOIN warehouse.dim_sprint s
            ON t.sprint_id = s.sprint_id
        WHERE t.status = 'Done'
    """).fetchdf()
)

In [ ]:
m23_deadline_check = con_dq.execute("""
    SELECT
        t.task_id,
        t.sprint_id,
        t.completed_date,
        s.end_date AS sprint_end_date,
        t.completed_on_time,
        CASE
            WHEN t.completed_date IS NULL THEN 'MISSING_COMPLETION_DATE'
            WHEN s.end_date IS NULL THEN 'MISSING_SPRINT_END_DATE'
            WHEN t.completed_date <= s.end_date THEN 'ON_TIME_BY_DATE'
            ELSE 'LATE_BY_DATE'
        END AS date_based_status
    FROM warehouse.fact_task t
    LEFT JOIN warehouse.dim_sprint s
        ON t.sprint_id = s.sprint_id
    WHERE t.status = 'Done'
    ORDER BY t.task_id
""").fetchdf()

print("Total Done tasks checked:", len(m23_deadline_check))
print("\nDate-based status counts:")
display(
    m23_deadline_check["date_based_status"]
    .value_counts(dropna=False)
    .rename_axis("date_based_status")
    .reset_index(name="task_count")
)

print("\nFlag versus date-based status:")
display(
    m23_deadline_check[
        (m23_deadline_check["date_based_status"] == "ON_TIME_BY_DATE")
        & (m23_deadline_check["completed_on_time"] != "Y")
        |
        (m23_deadline_check["date_based_status"] == "LATE_BY_DATE")
        & (m23_deadline_check["completed_on_time"] != "N")
    ].head(20)
)

In [ ]:
m23_mismatch_summary = con_dq.execute("""
    WITH checked AS (
        SELECT
            t.task_id,
            t.completed_on_time,
            t.completed_date,
            s.end_date AS sprint_end_date,
            CASE
                WHEN t.completed_date IS NULL OR s.end_date IS NULL
                    THEN 'UNKNOWN'
                WHEN t.completed_date <= s.end_date
                    THEN 'Y'
                ELSE 'N'
            END AS expected_flag
        FROM warehouse.fact_task t
        LEFT JOIN warehouse.dim_sprint s
            ON t.sprint_id = s.sprint_id
        WHERE t.status = 'Done'
    )
    SELECT
        COUNT(*) AS total_done_tasks,
        SUM(CASE
            WHEN completed_date IS NULL OR sprint_end_date IS NULL
            THEN 1 ELSE 0
        END) AS unknown_date_cases,
        SUM(CASE
            WHEN expected_flag IN ('Y', 'N')
             AND completed_on_time IS NOT NULL
             AND completed_on_time <> expected_flag
            THEN 1 ELSE 0
        END) AS flag_mismatches,
        SUM(CASE
            WHEN expected_flag IN ('Y', 'N')
             AND completed_on_time IS NULL
            THEN 1 ELSE 0
        END) AS missing_flags
    FROM checked
""").fetchdf()

display(m23_mismatch_summary)

In [ ]:
display(
    con_dq.execute("""
        SELECT
            t.task_id,
            t.team_id,
            t.sprint_id,
            t.completed_date,
            s.end_date AS sprint_end_date,
            t.completed_on_time AS stored_flag,
            CASE
                WHEN t.completed_date <= s.end_date THEN 'Y'
                ELSE 'N'
            END AS expected_flag
        FROM warehouse.fact_task t
        JOIN warehouse.dim_sprint s
            ON t.sprint_id = s.sprint_id
        WHERE t.status = 'Done'
          AND t.completed_date IS NOT NULL
          AND (
              (t.completed_date <= s.end_date AND t.completed_on_time <> 'Y')
              OR
              (t.completed_date > s.end_date AND t.completed_on_time <> 'N')
          )
    """).fetchdf()
)

In [ ]:
m23_corrected = con_dq.execute("""
    WITH team_sprint AS (
        SELECT
            t.team_id,
            t.sprint_id,
            COUNT(*) FILTER (
                WHERE t.status = 'Done'
            ) AS done_tasks,
            COUNT(*) FILTER (
                WHERE t.status = 'Done'
                  AND t.completed_date IS NOT NULL
                  AND s.end_date IS NOT NULL
            ) AS done_tasks_with_valid_dates,
            COUNT(*) FILTER (
                WHERE t.status = 'Done'
                  AND (
                      t.completed_date IS NULL
                      OR s.end_date IS NULL
                  )
            ) AS done_tasks_missing_dates,
            COUNT(*) FILTER (
                WHERE t.status = 'Done'
                  AND t.completed_date IS NOT NULL
                  AND s.end_date IS NOT NULL
                  AND t.completed_date <= s.end_date
            ) AS completed_on_time_by_date,
            COUNT(*) AS total_tasks
        FROM warehouse.fact_task t
        LEFT JOIN warehouse.dim_sprint s
            ON t.sprint_id = s.sprint_id
        GROUP BY t.team_id, t.sprint_id
    )
    SELECT
        *,
        completed_on_time_by_date::DOUBLE
            / NULLIF(done_tasks_with_valid_dates, 0)
            AS m23_on_time_rate,
        CASE
            WHEN total_tasks >= 5 THEN 'RELEASE'
            ELSE 'SUPPRESS'
        END AS privacy_status
    FROM team_sprint
    ORDER BY team_id, sprint_id
""").fetchdf()

print("Team-sprint groups:", len(m23_corrected))
print(
    "Done tasks missing dates:",
    int(m23_corrected["done_tasks_missing_dates"].sum())
)
print(
    "Total on-time tasks by actual dates:",
    int(m23_corrected["completed_on_time_by_date"].sum())
)
print("\nSample corrected M23 results:")
display(m23_corrected.head(10))

In [ ]:
# Use the already calculated M22/M25 results and corrected M23 results.
task_metric_validation = task_privacy_metrics[
    [
        "team_id",
        "sprint_id",
        "total_tasks",
        "planned_tasks",
        "done_tasks",
        "m22_completion_rate",
        "m25_avg_cycle_time_days",
        "invalid_cycle_time_tasks",
        "privacy_status"
    ]
].merge(
    m23_corrected[
        [
            "team_id",
            "sprint_id",
            "m23_on_time_rate",
            "done_tasks_missing_dates"
        ]
    ],
    on=["team_id", "sprint_id"],
    how="left",
    validate="one_to_one"
)

print("Team-sprint groups:", len(task_metric_validation))

print("\nPrivacy status counts:")
print(task_metric_validation["privacy_status"].value_counts())

print("\nMetric value completeness:")
display(
    task_metric_validation[
        [
            "m22_completion_rate",
            "m23_on_time_rate",
            "m25_avg_cycle_time_days"
        ]
    ].isna().sum().rename("missing_values").to_frame()
)

print("\nValidation issue totals:")
display(
    task_metric_validation[
        ["done_tasks_missing_dates", "invalid_cycle_time_tasks"]
    ].sum().to_frame("issue_count")
)

display(task_metric_validation.head(10))

In [ ]:
print("Risk table columns:")
print(
    con_dq.execute("""
        DESCRIBE warehouse.fact_attrition_risk
    """).fetchdf()["column_name"].tolist()
)

print("\nM27 definition:")
display(
    con_dq.execute("""
        SELECT *
        FROM source_db.raw.metric_dictionary
        WHERE metric_id = 'M27'
    """).fetchdf()
)

print("\nRisk-band distribution:")
display(
    con_dq.execute("""
        SELECT
            risk_band,
            COUNT(*) AS prediction_count,
            COUNT(DISTINCT employee_id) AS distinct_employees
        FROM warehouse.fact_attrition_risk
        GROUP BY risk_band
        ORDER BY risk_band
    """).fetchdf()
)

In [ ]:
m27_privacy_metrics = con_dq.execute("""
    WITH employee_risk AS (
        SELECT
            department_id,
            employee_id,
            risk_band,
            as_of_date,
            ROW_NUMBER() OVER (
                PARTITION BY employee_id
                ORDER BY as_of_date DESC, prediction_id DESC
            ) AS rn
        FROM warehouse.fact_attrition_risk
    ),
    latest_employee_risk AS (
        SELECT
            department_id,
            employee_id,
            risk_band,
            as_of_date
        FROM employee_risk
        WHERE rn = 1
    )
    SELECT
        department_id,
        MAX(as_of_date) AS as_of_date,
        COUNT(DISTINCT employee_id) AS group_size,
        COUNT(DISTINCT CASE
            WHEN risk_band = 'High' THEN employee_id
        END) AS high_risk_employees,
        COUNT(DISTINCT CASE
            WHEN risk_band = 'High' THEN employee_id
        END)::DOUBLE
            / NULLIF(COUNT(DISTINCT employee_id), 0)
            AS m27_high_risk_share,
        CASE
            WHEN COUNT(DISTINCT employee_id) >= 10 THEN 'RELEASE'
            ELSE 'SUPPRESS'
        END AS privacy_status
    FROM latest_employee_risk
    GROUP BY department_id
    ORDER BY department_id
""").fetchdf()

print("Departments:", len(m27_privacy_metrics))
print("\nPrivacy status counts:")
print(m27_privacy_metrics["privacy_status"].value_counts())

print("\nGroup-size summary:")
display(m27_privacy_metrics["group_size"].describe().to_frame("group_size"))

print("\nDepartment-level M27 results:")
display(m27_privacy_metrics)

In [ ]:
display(
    con_dq.execute("""
        SELECT
            metric_id,
            metric_name,
            formula,
            source_table,
            grain,
            minimum_group_size,
            caveat
        FROM source_db.raw.metric_dictionary
        WHERE metric_id IN (
            'M07', 'M08', 'M09', 'M10',
            'M11', 'M12', 'M15'
        )
        ORDER BY metric_id
    """).fetchdf()
)

In [ ]:
payroll_m10_m12 = con_dq.execute("""
    WITH employee_counts AS (
        SELECT
            DATE_TRUNC('month', period_end_date)::DATE AS month_start_date,
            department_id,
            COUNT(DISTINCT employee_id) AS employee_count,
            SUM(employer_cost) AS total_employer_cost,
            SUM(overtime_pay) AS total_overtime_pay,
            SUM(variable_pay) AS total_variable_pay,
            SUM(gross_pay) AS total_gross_pay
        FROM warehouse.fact_payroll
        GROUP BY 1, 2
    )
    SELECT
        month_start_date,
        department_id,
        employee_count,
        total_employer_cost,
        total_overtime_pay,
        total_variable_pay,
        total_gross_pay,

        CASE
            WHEN employee_count >= 5
            THEN total_employer_cost / NULLIF(employee_count, 0)
            ELSE NULL
        END AS m10_cost_per_head,

        CASE
            WHEN employee_count >= 5
            THEN 100.0 * total_overtime_pay
                 / NULLIF(total_employer_cost, 0)
            ELSE NULL
        END AS m11_overtime_share_pct,

        CASE
            WHEN employee_count >= 5
            THEN 100.0 * total_variable_pay
                 / NULLIF(total_gross_pay, 0)
            ELSE NULL
        END AS m12_variable_pay_share_pct,

        CASE
            WHEN employee_count >= 5 THEN 'RELEASE'
            ELSE 'SUPPRESS'
        END AS privacy_status

    FROM employee_counts
    ORDER BY month_start_date, department_id
""").fetchdf()

print("M10-M12 department-month records:", len(payroll_m10_m12))
print("\\nPrivacy status:")
display(
    payroll_m10_m12["privacy_status"]
    .value_counts()
    .rename_axis("privacy_status")
    .reset_index(name="groups")
)

print("\\nSample results:")
display(payroll_m10_m12.head(10))

In [ ]:
# Validate M10-M12 calculations and privacy suppression

released = payroll_m10_m12[
    payroll_m10_m12["privacy_status"] == "RELEASE"
].copy()

suppressed = payroll_m10_m12[
    payroll_m10_m12["privacy_status"] == "SUPPRESS"
].copy()

# 1. Check formulas for every released group
m10_error = (
    released["m10_cost_per_head"]
    - released["total_employer_cost"] / released["employee_count"]
).abs()

m11_expected = (
    100 * released["total_overtime_pay"]
    / released["total_employer_cost"].replace(0, float("nan"))
)
m11_error = (
    released["m11_overtime_share_pct"] - m11_expected
).abs()

m12_expected = (
    100 * released["total_variable_pay"]
    / released["total_gross_pay"].replace(0, float("nan"))
)
m12_error = (
    released["m12_variable_pay_share_pct"] - m12_expected
).abs()

# 2. Check that suppressed groups expose no calculated metrics
suppressed_values_hidden = suppressed[
    [
        "m10_cost_per_head",
        "m11_overtime_share_pct",
        "m12_variable_pay_share_pct"
    ]
].isna().all().all()

# 3. Display validation results
print("Released groups checked:", len(released))
print("M10 formula errors:", int((m10_error > 1e-8).sum()))
print("M11 formula errors:", int((m11_error > 1e-8).sum()))
print("M12 formula errors:", int((m12_error > 1e-8).sum()))
print("Suppressed metric values all hidden:", suppressed_values_hidden)

print("\nZero-denominator checks:")
print(
    "Zero employer-cost groups:",
    int((payroll_m10_m12["total_employer_cost"] == 0).sum())
)
print(
    "Zero gross-pay groups:",
    int((payroll_m10_m12["total_gross_pay"] == 0).sum())
)

print("\nValidation status:")
if (
    (m10_error <= 1e-8).all()
    and (m11_error <= 1e-8).all()
    and (m12_error <= 1e-8).all()
    and suppressed_values_hidden
):
    print("PASS — formulas and suppression checks passed.")
else:
    print("REVIEW REQUIRED — inspect the checks above.")

In [ ]:
payroll_recheck = con_dq.execute("""
    SELECT
        DATE_TRUNC('month', period_end_date)::DATE AS month_start_date,
        department_id,
        COUNT(DISTINCT employee_id) AS employee_count,
        SUM(gross_pay) AS gross_pay,
        SUM(net_pay) AS net_pay,
        SUM(employer_cost) AS employer_cost,
        SUM(lop_days) AS lop_days
    FROM warehouse.fact_payroll
    GROUP BY 1, 2
""").fetchdf()

# Compare against the previously calculated payroll privacy results
comparison = payroll_department_privacy_check.merge(
    payroll_recheck,
    on=["month_start_date", "department_id"],
    how="outer",
    suffixes=("_existing", "_recheck"),
    indicator=True
)

print("Department-month groups:", len(comparison))
print("Groups matched:", int((comparison["_merge"] == "both").sum()))
print("Groups missing from either result:", int((comparison["_merge"] != "both").sum()))

for metric in ["gross_pay", "net_pay", "employer_cost", "lop_days"]:
    difference = (
        comparison[f"{metric}_existing"] - comparison[f"{metric}_recheck"]
    ).abs()

    print(f"{metric} mismatches:", int((difference > 0.01).sum()))

print(
    "Employee-count mismatches:",
    int((
        comparison["employee_count_existing"]
        != comparison["employee_count_recheck"]
    ).sum())
)

print("\nSample comparison:")
display(comparison.head(10))

In [ ]:
print("Merge indicator counts:")
display(
    comparison["_merge"]
    .value_counts(dropna=False)
    .rename_axis("merge_status")
    .reset_index(name="groups")
)

print("\nRows not matched:")
display(
    comparison.loc[
        comparison["_merge"] != "both",
        ["month_start_date", "department_id", "_merge"]
    ]
)

print("\nM07-M09 and M15 validation:")
if (
    (comparison["_merge"] == "both").all()
    and len(comparison) == 198
):
    print("PASS — all 198 department-month groups matched.")
else:
    print("REVIEW REQUIRED — investigate unmatched groups.")

In [ ]:
# Recalculate M05 and M06 privacy status using unrounded rolling averages

rolling_privacy = con_dq.execute("""
    WITH monthly AS (
        SELECT
            month_start_date,
            department_id,
            headcount_end,
            exits,
            voluntary_exits,
            COUNT(*) OVER (
                PARTITION BY department_id
                ORDER BY month_start_date
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS months_in_window,
            AVG(headcount_end) OVER (
                PARTITION BY department_id
                ORDER BY month_start_date
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ) AS avg_headcount_12m
        FROM warehouse.fact_headcount_monthly
    )
    SELECT
        month_start_date,
        department_id,
        months_in_window,
        avg_headcount_12m
    FROM monthly
""").fetchdf()

# Replace the rounded proxy with the unrounded rolling average.
# Incomplete windows remain suppressed.
for metric_id in ["M05", "M06"]:
    mask = department_month_metrics["metric_id"] == metric_id

    updated = department_month_metrics.loc[
        mask, ["month_start_date", "department_id"]
    ].merge(
        rolling_privacy,
        on=["month_start_date", "department_id"],
        how="left",
        validate="one_to_one"
    )

    department_month_metrics.loc[mask, "group_size"] = (
        updated["avg_headcount_12m"].to_numpy()
    )

    release_mask = (
        (updated["months_in_window"].to_numpy() == 12)
        & (updated["avg_headcount_12m"].to_numpy() >= 10)
    )

    department_month_metrics.loc[mask, "privacy_status"] = [
        "RELEASE" if allowed else "SUPPRESS"
        for allowed in release_mask
    ]

    # Hide metric values whenever the group is suppressed.
    suppressed_indices = department_month_metrics.index[
        mask & (department_month_metrics["privacy_status"] == "SUPPRESS")
    ]
    department_month_metrics.loc[suppressed_indices, "metric_value"] = None

print("Updated M05-M06 privacy summary:")
display(
    department_month_metrics[
        department_month_metrics["metric_id"].isin(["M05", "M06"])
    ].groupby(["metric_id", "privacy_status"])
    .size()
    .unstack(fill_value=0)
)

print("\nRolling-window completeness:")
display(
    rolling_privacy["months_in_window"]
    .value_counts()
    .sort_index()
    .rename_axis("months_in_window")
    .reset_index(name="department_month_groups")
)

In [ ]:
# Fix the integer dtype before assigning rolling averages
department_month_metrics["group_size"] = (
    department_month_metrics["group_size"].astype("float64")
)

for metric_id in ["M05", "M06"]:
    mask = department_month_metrics["metric_id"] == metric_id

    updated = department_month_metrics.loc[
        mask, ["month_start_date", "department_id"]
    ].merge(
        rolling_privacy,
        on=["month_start_date", "department_id"],
        how="left",
        validate="one_to_one"
    )

    # Keep the full-precision rolling average
    department_month_metrics.loc[mask, "group_size"] = (
        updated["avg_headcount_12m"].to_numpy(dtype="float64")
    )

    # Require a complete 12-month window and average headcount >= 10
    release_mask = (
        (updated["months_in_window"].to_numpy() == 12)
        & (updated["avg_headcount_12m"].to_numpy() >= 10)
    )

    department_month_metrics.loc[mask, "privacy_status"] = [
        "RELEASE" if allowed else "SUPPRESS"
        for allowed in release_mask
    ]

    # Suppress metric values for all non-releasable groups
    suppressed_indices = department_month_metrics.index[
        mask
        & (department_month_metrics["privacy_status"] == "SUPPRESS")
    ]

    department_month_metrics.loc[
        suppressed_indices, "metric_value"
    ] = float("nan")

print("Updated M05-M06 privacy summary:")
display(
    department_month_metrics[
        department_month_metrics["metric_id"].isin(["M05", "M06"])
    ]
    .groupby(["metric_id", "privacy_status"])
    .size()
    .unstack(fill_value=0)
)

print("\nRolling-window completeness:")
display(
    rolling_privacy["months_in_window"]
    .value_counts()
    .sort_index()
    .rename_axis("months_in_window")
    .reset_index(name="department_month_groups")
)

In [ ]:
rolling_metrics = department_month_metrics[
    department_month_metrics["metric_id"].isin(["M05", "M06"])
].copy()

privacy_check = (
    rolling_metrics
    .groupby(["metric_id", "privacy_status"])
    .agg(
        groups=("department_id", "size"),
        visible_values=("metric_value", lambda s: s.notna().sum()),
        hidden_values=("metric_value", lambda s: s.isna().sum())
    )
    .reset_index()
)

display(privacy_check)

suppressed_values_hidden = (
    rolling_metrics.loc[
        rolling_metrics["privacy_status"] == "SUPPRESS",
        "metric_value"
    ].isna().all()
)

print("\nAll suppressed M05/M06 values hidden:", suppressed_values_hidden)

In [ ]:
print("Metric IDs currently validated:")
validated_metric_ids = sorted(
    set(department_month_metrics["metric_id"].dropna().astype(str))
)

print(validated_metric_ids)
print("Count:", len(validated_metric_ids))

print("\nMetric IDs available in the source catalog:")
display(
    con_dq.execute("""
        SELECT
            metric_id,
            metric_name,
            grain,
            minimum_group_size
        FROM source_db.raw.metric_dictionary
        ORDER BY metric_id
    """).fetchdf()
)

In [ ]:
# Build an explicit audit of threshold-defined metrics

catalog_metrics = con_dq.execute("""
    SELECT
        metric_id,
        metric_name,
        grain,
        minimum_group_size
    FROM source_db.raw.metric_dictionary
    WHERE minimum_group_size IS NOT NULL
      AND TRIM(CAST(minimum_group_size AS VARCHAR)) <> ''
      AND LOWER(TRIM(CAST(minimum_group_size AS VARCHAR))) <> 'nan'
    ORDER BY metric_id
""").fetchdf()

validated_ids = {
    "M01", "M02", "M03", "M04", "M05", "M06",
    "M07", "M08", "M09", "M10", "M11", "M12",
    "M13", "M14", "M15",
    "M22", "M23", "M25", "M27"
}

catalog_metrics["validation_status"] = catalog_metrics["metric_id"].apply(
    lambda metric_id: (
        "Validated separately — confirm evidence"
        if metric_id in validated_ids
        else "Pending"
    )
)

print("Threshold-defined catalog metrics:", len(catalog_metrics))
print("Metric IDs with validation evidence to consolidate:", len(validated_ids))
print("\nAudit:")
display(catalog_metrics)

print("\nMetrics with no defined minimum group size:")
display(
    con_dq.execute("""
        SELECT metric_id, metric_name, grain
        FROM source_db.raw.metric_dictionary
        WHERE minimum_group_size IS NULL
           OR TRIM(CAST(minimum_group_size AS VARCHAR)) = ''
           OR LOWER(TRIM(CAST(minimum_group_size AS VARCHAR))) = 'nan'
        ORDER BY metric_id
    """).fetchdf()
)

In [ ]:
c14_current = con_dq.execute("""
    SELECT *
    FROM warehouse.dq_results
    WHERE check_id = 'DQ-C14'
""").fetchdf()

display(c14_current)

In [ ]:
c14_reason = (
    "PARTIAL: Validated privacy calculations for 19 catalog metrics with "
    "defined minimum-group-size thresholds across workforce, payroll, "
    "attendance, task, and attrition-risk metric families. "
    "M05-M06 use unrounded rolling 12-month average headcount: "
    "539 RELEASE and 352 SUPPRESS per metric; incomplete windows and "
    "groups below threshold are suppressed. Payroll M07-M12 and M15 "
    "calculations and reconciliation checks passed; attendance and task "
    "metrics were calculated and privacy-checked. "
    "Nine catalog metrics (M16-M21, M24, M26, M28) have no defined "
    "minimum-group-size threshold. Metric-specific policies and evidence "
    "still require final consolidation and review."
)

con_dq.execute("""
    UPDATE warehouse.dq_results
    SET
        status = 'PARTIAL',
        failing_rows = NULL,
        reason = ?,
        execution_status = 'PARTIAL',
        interpretation = 'Partially implemented; catalog gaps and final evidence review remain'
    WHERE check_id = 'DQ-C14'
""", [c14_reason])

display(
    con_dq.execute("""
        SELECT *
        FROM warehouse.dq_results
        WHERE check_id = 'DQ-C14'
    """).fetchdf()
)

In [ ]:
display(
    con_dq.execute("""
        SELECT
            check_id,
            check_name,
            severity,
            status,
            failing_rows,
            execution_status
        FROM warehouse.dq_results
        ORDER BY check_id
    """).fetchdf()
)

print("\nStatus counts:")
display(
    con_dq.execute("""
        SELECT status, COUNT(*) AS check_count
        FROM warehouse.dq_results
        GROUP BY status
        ORDER BY status
    """).fetchdf()
)

In [ ]:
from pathlib import Path
import pandas as pd

coverage_path = (
    Path.cwd()
    / "04_Validation_Vault"
    / "PQ-03_planted_issue_coverage.csv"
)

print("Current working directory:", Path.cwd())
print("Coverage report exists:", coverage_path.exists())

if coverage_path.exists():
    coverage = pd.read_csv(coverage_path)
    display(coverage)

    print("\nNumber of planted issues:", len(coverage))
else:
    print("Report not found at this path.")
    print("Search the project folder for PQ-03_planted_issue_coverage.csv.")

In [ ]:
import duckdb
from pathlib import Path

db_path = Path(
    r"C:\Users\Hp\Documents\pulseiq\05_Starter\pulseiq_warehouse.duckdb"
)

print("Database exists:", db_path.exists())

con = duckdb.connect(str(db_path), read_only=True)

print("Connected to PulseIQ warehouse successfully!")

In [ ]:
# Check whether dq_results exists in the warehouse
tables = con.execute("""
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'warehouse'
      AND table_name = 'dq_results'
""").df()

print(tables)

In [ ]:
print(con.execute("""
    DESCRIBE warehouse.dq_results
""").df())

In [ ]:
dq_df = con.execute("""
    SELECT *
    FROM warehouse.dq_results
    ORDER BY severity, check_id
""").df()

print("DQ results shape:", dq_df.shape)
display(dq_df)

In [ ]:
print("Total DQ checks:", len(dq_df))

print("\nStatus summary:")
display(
    dq_df["status"]
    .fillna("NULL")
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

print("\nExecution status summary:")
display(
    dq_df["execution_status"]
    .fillna("NULL")
    .value_counts()
    .rename_axis("execution_status")
    .reset_index(name="count")
)

print("\nSeverity summary:")
display(
    dq_df["severity"]
    .fillna("NULL")
    .value_counts()
    .rename_axis("severity")
    .reset_index(name="count")
)

In [ ]:
import pandas as pd
from pathlib import Path

# Summarize recorded DQ results
dq_summary = (
    dq_df.groupby(
        ["severity", "status", "execution_status"],
        dropna=False
    )
    .size()
    .reset_index(name="check_count")
)

print("DQ RESULTS SUMMARY")
print("==================")
print("Total checks:", len(dq_df))
print("Passed:", (dq_df["status"] == "PASS").sum())
print("Failed:", (dq_df["status"] == "FAIL").sum())
print("Blocked:", (dq_df["status"] == "BLOCKED").sum())
print("Partial:", (dq_df["status"] == "PARTIAL").sum())

display(dq_summary)

# Save evidence
output_path = Path(
    r"C:\Users\Hp\Documents\pulseiq\Task3_DQ_Summary.csv"
)

dq_summary.to_csv(output_path, index=False)
print("Saved summary:", output_path)
print("File exists:", output_path.exists())

In [ ]:
dq_summary.to_csv(output_path, index=False)
print("Saved summary:", output_path)
print("File exists:", output_path.exists())

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

# Connect to the PulseIQ warehouse
db_path = Path("05_Starter/pulseiq_warehouse.duckdb")

print("Database exists:", db_path.exists())

con = duckdb.connect(str(db_path.resolve()), read_only=True)

# Load recorded DQ results
dq_df = con.execute("""
    SELECT *
    FROM warehouse.dq_results
    ORDER BY severity, check_id
""").df()

print("DQ results shape:", dq_df.shape)
print("Total DQ checks:", len(dq_df))

# Create summary
dq_summary = (
    dq_df.groupby(
        ["severity", "status", "execution_status"],
        dropna=False
    )
    .size()
    .reset_index(name="check_count")
)

display(dq_summary)

# Save summary CSV
output_path = Path("Task3_DQ_Summary.csv")
dq_summary.to_csv(output_path, index=False)

print("Saved summary:", output_path.resolve())
print("File exists:", output_path.exists())
print("File size:", output_path.stat().st_size, "bytes")

con.close()